# Phase 2 — Data Profiling & Audit

**Objective:** Understand the raw dataset completely before making any changes.

**Important:** This notebook is **read-only** with respect to the raw dataset. It does not rename, convert, fill, standardize, delete, or overwrite raw data.

## 1. Objective

Profile and audit the raw Google Ads sales dataset, quantify data-quality issues, and produce evidence for the Phase 3 cleaning specification.

In [18]:
from pathlib import Path
import hashlib
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Resolve the raw CSV from the notebook/project location.
raw_filename = "GoogleAds_DataAnalytics_Sales_Uncleaned.csv"
workspace_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path.home() / "Desktop" / "project1" / "google-ads-marketing-performance",
]
candidate_paths = [
    base / "data" / "raw" / raw_filename
    for base in workspace_candidates
]
raw_path = next((p for p in candidate_paths if p.is_file()), None)

if raw_path is None:
    searched_paths = "\n".join(str(p) for p in candidate_paths)
    raise FileNotFoundError(
        f"Raw CSV not found. Searched:\n{searched_paths}"
    )

raw_path = raw_path.resolve()
print("Raw file:", raw_path)
print("Exists:", raw_path.exists())
print("Filename:", raw_path.name)

# Integrity reference recorded for the raw input.
sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()
print("SHA-256:", sha256)

Raw file: C:\Users\mahes\Desktop\project1\google-ads-marketing-performance\data\raw\GoogleAds_DataAnalytics_Sales_Uncleaned.csv
Exists: True
Filename: GoogleAds_DataAnalytics_Sales_Uncleaned.csv
SHA-256: 0f947d4e7fb1eb2b54534e42182ce752dd1f6930b2c2784f7f22378b475388bb


## 2. Load Raw Data

Load the raw CSV without permanently changing its values or schema.

In [19]:
df = pd.read_csv(raw_path, low_memory=False)

print(f"Loaded successfully: {df is not None}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

Loaded successfully: True
Rows: 2,600
Columns: 13

First 5 rows:


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
0,A1000,DataAnalyticsCourse,104.0000,"4,498.0000",$231.88,14.0000,7.0000,0.0580,$1892,2024-11-16,hyderabad,desktop,learn data analytics
1,A1001,DataAnalyticsCourse,173.0000,"5,107.0000",$216.84,10.0000,8.0000,0.0460,$1679,20-11-2024,hyderabad,mobile,data analytics course
2,A1002,Data Anlytics Corse,90.0000,"4,544.0000",$203.66,26.0000,9.0000,NaN,$1624,2024/11/16,hyderabad,Desktop,data analitics online
3,A1003,Data Analytcis Course,142.0000,"3,185.0000",$237.66,17.0000,6.0000,NaN,$1225,2024-11-26,HYDERABAD,tablet,data anaytics training
4,A1004,Data Analytics Corse,156.0000,"3,361.0000",$195.9,30.0000,8.0000,NaN,$1091,2024-11-22,hyderabad,desktop,online data analytic



Last 5 rows:


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
2595,A3595,DataAnalyticsCourse,88.0000,"5,344.0000",$242.07,17.0000,9.0000,0.0540,$1418,29-11-2024,HYDERABAD,MOBILE,online data analytic
2596,A3596,DataAnalyticsCourse,154.0000,"3,211.0000",$248.28,14.0000,6.0000,0.0390,$1950,2024/11/28,hyderabad,TABLET,data analitics online
2597,A3597,Data Anlytics Corse,113.0000,"3,808.0000",$233.25,18.0000,4.0000,0.0350,$1085,2024-11-02,Hyderbad,desktop,data anaytics training
2598,A3598,Data Analytics Corse,196.0000,"5,853.0000",$220.13,16.0000,7.0000,0.0360,$1558,2024-11-08,hydrebad,Tablet,data anaytics training
2599,A3599,Data Analytics Corse,NaN,"5,453.0000",NaN,12.0000,5.0000,NaN,$1174,2024/11/22,HYDERABAD,desktop,analytics for data


## 3. Create the Dataset Profile

The purpose of this step is to create a complete structural profile of the raw dataset before any cleaning or transformation is performed.

For every raw column, the profile will capture:

- Column name
- Data type
- Non-null count
- Missing count
- Missing percentage
- Unique count

This profiling table provides the baseline for the subsequent data-quality audits.

**Important:** The raw DataFrame `df` will not be modified in this step. The profile is created from the existing raw data only.

In [20]:
# Step 3 — Create the dataset profile

data_profile = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isna().sum().values,
    "Missing_%": (df.isna().mean() * 100).values,
    "Unique_Count": df.nunique(dropna=True).values
})

display(data_profile)

,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_%,Unique_Count
0,Ad_ID,object,2600,0,0.0000,2600
1,Campaign_Name,object,2600,0,0.0000,4
2,Clicks,float64,2488,112,4.3077,120
3,Impressions,float64,2546,54,2.0769,1702
4,Cost,object,2503,97,3.7308,2106
5,Leads,float64,2552,48,1.8462,21
6,Conversions,float64,2526,74,2.8462,8
7,Conversion Rate,float64,1974,626,24.0769,105
8,Sale_Amount,object,2461,139,5.3462,921
9,Ad_Date,object,2600,0,0.0000,90


In [21]:
# Validate the dataset profile

print("Dataset profile validation")
print("-" * 40)

print(f"Profile rows: {data_profile.shape[0]}")
print(f"Dataset columns: {df.shape[1]}")
print(f"Dataset rows: {df.shape[0]:,}")

print(
    "\nAll columns included:",
    data_profile["Column"].tolist() == df.columns.tolist()
)

print(
    "Non-null + missing counts reconcile:",
    (
        data_profile["Non_Null_Count"]
        + data_profile["Missing_Count"]
    ).eq(len(df)).all()
)

Dataset profile validation
----------------------------------------
Profile rows: 13
Dataset columns: 13
Dataset rows: 2,600

All columns included: True
Non-null + missing counts reconcile: True


## 4. Audit Column Names

The purpose of this step is to record the exact column names present in the raw dataset and compare them with the planned analytical schema.

The raw column names will **not be renamed or modified** during Phase 2.

The audit will identify:

- Exact raw column names
- Expected analytical column names
- Naming matches
- Naming inconsistencies

**Phase 2 action:** Identify and document naming inconsistencies.

**Phase 3 action:** Rename columns according to the approved analytical schema.

In [22]:
# Step 4 — Audit column names

# Planned analytical schema
planned_columns = [
    "Ad_ID",
    "Campaign_Name",
    "Clicks",
    "Impressions",
    "Cost",
    "Leads",
    "Conversions",
    "Conversion_Rate",
    "Sale_Amount",
    "Ad_Date",
    "Location",
    "Device",
    "Keyword"
]

# Record exact raw column names
raw_columns = df.columns.tolist()

print("Exact raw column names:")
print("-" * 40)

for i, column in enumerate(raw_columns, start=1):
    print(f"{i:>2}. {column}")

print(f"\nNumber of raw columns: {len(raw_columns)}")
print(f"Number of planned columns: {len(planned_columns)}")

Exact raw column names:
----------------------------------------
 1. Ad_ID
 2. Campaign_Name
 3. Clicks
 4. Impressions
 5. Cost
 6. Leads
 7. Conversions
 8. Conversion Rate
 9. Sale_Amount
10. Ad_Date
11. Location
12. Device
13. Keyword

Number of raw columns: 13
Number of planned columns: 13


In [23]:
# Compare raw column names with the planned analytical schema

column_name_audit = pd.DataFrame({
    "Position": range(1, len(planned_columns) + 1),
    "Raw_Column": raw_columns,
    "Planned_Column": planned_columns
})

column_name_audit["Name_Match"] = (
    column_name_audit["Raw_Column"]
    == column_name_audit["Planned_Column"]
)

display(column_name_audit)

,Position,Raw_Column,Planned_Column,Name_Match
0,1,Ad_ID,Ad_ID,True
1,2,Campaign_Name,Campaign_Name,True
2,3,Clicks,Clicks,True
3,4,Impressions,Impressions,True
4,5,Cost,Cost,True
5,6,Leads,Leads,True
6,7,Conversions,Conversions,True
7,8,Conversion Rate,Conversion_Rate,False
8,9,Sale_Amount,Sale_Amount,True
9,10,Ad_Date,Ad_Date,True


In [24]:
# Identify naming inconsistencies

naming_issues = column_name_audit[
    ~column_name_audit["Name_Match"]
].copy()

print(f"Naming inconsistencies found: {len(naming_issues)}")

if len(naming_issues) > 0:
    print("\nNaming inconsistencies:")
    display(naming_issues)
else:
    print("No naming inconsistencies found.")

Naming inconsistencies found: 1

Naming inconsistencies:


,Position,Raw_Column,Planned_Column,Name_Match
7,8,Conversion Rate,Conversion_Rate,False


## 5. Audit Data Types

The purpose of this step is to audit the data type of every raw column and classify each column according to its analytical role.

Each column will be classified as one of the following:

- Numeric
- Categorical
- Date
- Currency
- Identifier

The audit will also identify potential data-type problems, including:

- Currency fields stored as text
- Date fields stored as text
- Numeric fields containing non-numeric values
- Numeric fields containing unexpected formatting

**Important:** No permanent data-type conversion will be performed in this step. Any parsing or conversion used for investigation will be temporary and will not modify the raw DataFrame `df`.

In [25]:
# Step 5 — Audit data types

# Analytical classification based on the planned schema
type_classification = {
    "Ad_ID": "Identifier",
    "Campaign_Name": "Categorical",
    "Clicks": "Numeric",
    "Impressions": "Numeric",
    "Cost": "Currency",
    "Leads": "Numeric",
    "Conversions": "Numeric",
    "Conversion Rate": "Numeric",
    "Sale_Amount": "Currency",
    "Ad_Date": "Date",
    "Location": "Categorical",
    "Device": "Categorical",
    "Keyword": "Categorical"
}

data_type_audit = pd.DataFrame({
    "Column": df.columns,
    "Raw_Data_Type": df.dtypes.astype(str).values,
    "Analytical_Classification": [
        type_classification.get(column, "Review Required")
        for column in df.columns
    ],
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isna().sum().values
})

display(data_type_audit)

,Column,Raw_Data_Type,Analytical_Classification,Non_Null_Count,Missing_Count
0,Ad_ID,object,Identifier,2600,0
1,Campaign_Name,object,Categorical,2600,0
2,Clicks,float64,Numeric,2488,112
3,Impressions,float64,Numeric,2546,54
4,Cost,object,Currency,2503,97
5,Leads,float64,Numeric,2552,48
6,Conversions,float64,Numeric,2526,74
7,Conversion Rate,float64,Numeric,1974,626
8,Sale_Amount,object,Currency,2461,139
9,Ad_Date,object,Date,2600,0


In [26]:
# Investigate potential data-type problems
# Temporary parsing only — df itself is not modified.

def temporary_numeric(series):
    """
    Temporarily remove common currency formatting and attempt
    numeric parsing. The original series remains unchanged.
    """
    return pd.to_numeric(
        series.astype("string")
        .str.replace(r"[$,]", "", regex=True)
        .str.strip(),
        errors="coerce"
    )


numeric_columns = [
    "Clicks",
    "Impressions",
    "Leads",
    "Conversions",
    "Conversion Rate"
]

currency_columns = [
    "Cost",
    "Sale_Amount"
]

date_column = "Ad_Date"

type_issue_results = []

# Numeric fields
for column in numeric_columns:
    parsed = temporary_numeric(df[column])
    
    non_numeric_count = int(
        df[column].notna().sum() - parsed.notna().sum()
    )
    
    type_issue_results.append({
        "Column": column,
        "Classification": "Numeric",
        "Raw_Data_Type": str(df[column].dtype),
        "Non_Null": int(df[column].notna().sum()),
        "Temporarily_Parseable": int(parsed.notna().sum()),
        "Non_Numeric_Non_Missing": non_numeric_count
    })

# Currency fields
for column in currency_columns:
    parsed = temporary_numeric(df[column])
    
    non_numeric_count = int(
        df[column].notna().sum() - parsed.notna().sum()
    )
    
    type_issue_results.append({
        "Column": column,
        "Classification": "Currency",
        "Raw_Data_Type": str(df[column].dtype),
        "Non_Null": int(df[column].notna().sum()),
        "Temporarily_Parseable": int(parsed.notna().sum()),
        "Non_Numeric_Non_Missing": non_numeric_count
    })

# Date field
parsed_dates = pd.to_datetime(
    df[date_column].astype("string"),
    errors="coerce",
    format="mixed"
)

date_non_parseable = int(
    df[date_column].notna().sum() - parsed_dates.notna().sum()
)

type_issue_results.append({
    "Column": date_column,
    "Classification": "Date",
    "Raw_Data_Type": str(df[date_column].dtype),
    "Non_Null": int(df[date_column].notna().sum()),
    "Temporarily_Parseable": int(parsed_dates.notna().sum()),
    "Non_Numeric_Non_Missing": date_non_parseable
})

type_issue_audit = pd.DataFrame(type_issue_results)

display(type_issue_audit)

,Column,Classification,Raw_Data_Type,Non_Null,Temporarily_Parseable,Non_Numeric_Non_Missing
0,Clicks,Numeric,float64,2488,2488,0
1,Impressions,Numeric,float64,2546,2546,0
2,Leads,Numeric,float64,2552,2552,0
3,Conversions,Numeric,float64,2526,2526,0
4,Conversion Rate,Numeric,float64,1974,1974,0
5,Cost,Currency,object,2503,2503,0
6,Sale_Amount,Currency,object,2461,2461,0
7,Ad_Date,Date,object,2600,2600,0


In [27]:
# Step 5 summary

print("DATA TYPE AUDIT SUMMARY")
print("-" * 50)

for classification in [
    "Identifier",
    "Categorical",
    "Numeric",
    "Currency",
    "Date"
]:
    count = int(
        (data_type_audit["Analytical_Classification"] == classification).sum()
    )
    print(f"{classification}: {count} column(s)")

print("\nRaw data types:")
display(
    data_type_audit[
        ["Column", "Raw_Data_Type", "Analytical_Classification"]
    ]
)

print("\nPotential parsing issues:")
display(
    type_issue_audit[
        type_issue_audit["Non_Numeric_Non_Missing"] > 0
    ]
)

DATA TYPE AUDIT SUMMARY
--------------------------------------------------
Identifier: 1 column(s)
Categorical: 4 column(s)
Numeric: 5 column(s)
Currency: 2 column(s)
Date: 1 column(s)

Raw data types:


,Column,Raw_Data_Type,Analytical_Classification
0,Ad_ID,object,Identifier
1,Campaign_Name,object,Categorical
2,Clicks,float64,Numeric
3,Impressions,float64,Numeric
4,Cost,object,Currency
5,Leads,float64,Numeric
6,Conversions,float64,Numeric
7,Conversion Rate,float64,Numeric
8,Sale_Amount,object,Currency
9,Ad_Date,object,Date



Potential parsing issues:


,Column,Classification,Raw_Data_Type,Non_Null,Temporarily_Parseable,Non_Numeric_Non_Missing


## 6. Perform the Missing-Value Audit

The purpose of this step is to quantify missing values in the raw dataset before any cleaning or imputation is performed.

The audit will calculate:

- Missing count per column
- Missing percentage per column
- Total missing cells

The columns with the highest number of missing values will also be identified for further investigation.

**Important:** Missing values will not be filled, removed, or otherwise modified in this step. The raw DataFrame `df` remains unchanged.

The project specification indicates approximately 1,150 missing cells. This value will be verified against the actual raw dataset rather than assumed.

In [28]:
# Step 6 — Missing-value audit

missing_value_audit = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isna().sum().values,
    "Missing_%": (df.isna().mean() * 100).values,
    "Non_Missing_Count": df.notna().sum().values
})

# Sort by missing count so the most affected columns appear first
missing_value_audit = missing_value_audit.sort_values(
    by="Missing_Count",
    ascending=False
).reset_index(drop=True)

display(missing_value_audit)

,Column,Missing_Count,Missing_%,Non_Missing_Count
0,Conversion Rate,626,24.0769,1974
1,Sale_Amount,139,5.3462,2461
2,Clicks,112,4.3077,2488
3,Cost,97,3.7308,2503
4,Conversions,74,2.8462,2526
5,Impressions,54,2.0769,2546
6,Leads,48,1.8462,2552
7,Campaign_Name,0,0.0000,2600
8,Ad_ID,0,0.0000,2600
9,Ad_Date,0,0.0000,2600


In [29]:
# Calculate total missing cells

total_missing_cells = int(df.isna().sum().sum())

print(f"Total missing cells: {total_missing_cells:,}")

Total missing cells: 1,150


In [30]:
# Identify columns with the highest number of missing values

print("Columns most affected by missing values:")
display(
    missing_value_audit[
        missing_value_audit["Missing_Count"] > 0
    ]
)

Columns most affected by missing values:


,Column,Missing_Count,Missing_%,Non_Missing_Count
0,Conversion Rate,626,24.0769,1974
1,Sale_Amount,139,5.3462,2461
2,Clicks,112,4.3077,2488
3,Cost,97,3.7308,2503
4,Conversions,74,2.8462,2526
5,Impressions,54,2.0769,2546
6,Leads,48,1.8462,2552


In [31]:
# Validate the missing-value audit

print("MISSING-VALUE AUDIT VALIDATION")
print("-" * 45)

print(f"Dataset rows: {len(df):,}")
print(f"Dataset columns: {len(df.columns):,}")
print(f"Total missing cells: {total_missing_cells:,}")

print(
    "\nMissing counts reconcile:",
    missing_value_audit["Missing_Count"].sum() == total_missing_cells
)

print(
    "Missing percentages calculated correctly:",
    np.allclose(
        missing_value_audit["Missing_%"].values,
        (
            missing_value_audit["Missing_Count"]
            / len(df)
            * 100
        ).values
    )
)

MISSING-VALUE AUDIT VALIDATION
---------------------------------------------
Dataset rows: 2,600
Dataset columns: 13
Total missing cells: 1,150

Missing counts reconcile: True
Missing percentages calculated correctly: True


## 7. Investigate Missing-Value Meaning

The purpose of this step is to investigate why important fields contain missing values and whether the missingness appears to represent a structural zero, an unknown value, a potential data error, or a situation requiring a business rule.

The following relationships will be investigated:

- `Conversions` ↔ `Sale_Amount`
- `Conversions` ↔ `Leads`
- `Clicks` ↔ `Impressions`
- `Cost` ↔ `Clicks`

Specific missing-value patterns will also be examined:

- `Sale_Amount` missing + `Conversions = 0`
- `Sale_Amount` missing + `Conversions > 0`
- `Conversions` missing + `Sale_Amount > 0`
- `Cost` missing
- `Clicks` missing
- `Impressions` missing
- `Leads` missing

**Important:** Missing values will not be filled, removed, or replaced in this step.

The classifications in this audit are investigative conclusions only. A final treatment must be supported by an approved business rule before Phase 3 cleaning.

In [32]:
# Step 7 — Investigate missing-value meaning
# Temporary parsing only. The raw DataFrame df is NOT modified.

def temporary_numeric(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(r"[$,]", "", regex=True)
        .str.strip(),
        errors="coerce"
    )

investigation = df.copy()

investigation["__Clicks"] = temporary_numeric(df["Clicks"])
investigation["__Impressions"] = temporary_numeric(df["Impressions"])
investigation["__Leads"] = temporary_numeric(df["Leads"])
investigation["__Conversions"] = temporary_numeric(df["Conversions"])
investigation["__Cost"] = temporary_numeric(df["Cost"])
investigation["__Sale_Amount"] = temporary_numeric(df["Sale_Amount"])

print("Temporary investigation dataset created.")
print(f"Rows: {len(investigation):,}")
print("Raw df modified: No")

Temporary investigation dataset created.
Rows: 2,600
Raw df modified: No


In [33]:
# Investigate the required missing-value patterns

sale_missing_conversions_zero = (
    df["Sale_Amount"].isna()
    & investigation["__Conversions"].eq(0)
)

sale_missing_conversions_positive = (
    df["Sale_Amount"].isna()
    & investigation["__Conversions"].gt(0)
)

conversions_missing_sale_positive = (
    df["Conversions"].isna()
    & investigation["__Sale_Amount"].gt(0)
)

cost_missing = df["Cost"].isna()
clicks_missing = df["Clicks"].isna()
impressions_missing = df["Impressions"].isna()
leads_missing = df["Leads"].isna()

missing_pattern_audit = pd.DataFrame({
    "Pattern": [
        "Sale_Amount missing + Conversions = 0",
        "Sale_Amount missing + Conversions > 0",
        "Conversions missing + Sale_Amount > 0",
        "Cost missing",
        "Clicks missing",
        "Impressions missing",
        "Leads missing"
    ],
    "Records_Affected": [
        int(sale_missing_conversions_zero.sum()),
        int(sale_missing_conversions_positive.sum()),
        int(conversions_missing_sale_positive.sum()),
        int(cost_missing.sum()),
        int(clicks_missing.sum()),
        int(impressions_missing.sum()),
        int(leads_missing.sum())
    ]
})

display(missing_pattern_audit)

,Pattern,Records_Affected
0,Sale_Amount missing + Conversions = 0,0
1,Sale_Amount missing + Conversions > 0,135
2,Conversions missing + Sale_Amount > 0,70
3,Cost missing,97
4,Clicks missing,112
5,Impressions missing,54
6,Leads missing,48


In [34]:
# Inspect records for each important missing-value pattern

patterns = {
    "Sale_Amount missing + Conversions = 0":
        sale_missing_conversions_zero,

    "Sale_Amount missing + Conversions > 0":
        sale_missing_conversions_positive,

    "Conversions missing + Sale_Amount > 0":
        conversions_missing_sale_positive,

    "Cost missing":
        cost_missing,

    "Clicks missing":
        clicks_missing,

    "Impressions missing":
        impressions_missing,

    "Leads missing":
        leads_missing
}

display_columns = [
    "Ad_ID",
    "Campaign_Name",
    "Clicks",
    "Impressions",
    "Cost",
    "Leads",
    "Conversions",
    "Conversion Rate",
    "Sale_Amount",
    "Ad_Date",
    "Location",
    "Device",
    "Keyword"
]

for pattern_name, mask in patterns.items():
    print("\n" + "=" * 70)
    print(f"{pattern_name}")
    print(f"Records affected: {int(mask.sum())}")
    print("=" * 70)
    
    if mask.any():
        display(df.loc[mask, display_columns].head(20))
    else:
        print("No records found.")


Sale_Amount missing + Conversions = 0
Records affected: 0
No records found.

Sale_Amount missing + Conversions > 0
Records affected: 135


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
34,A1034,Data Analytcis Course,121.0000,"5,610.0000",$214.96,23.0000,8.0000,NaN,NaN,2024-11-25,hyderabad,Mobile,data analytics course
59,A1059,Data Analytics Corse,179.0000,"3,025.0000",$238.81,17.0000,3.0000,0.0170,NaN,2024-11-11,HYDERABAD,mobile,data analitics online
72,A1072,DataAnalyticsCourse,133.0000,"3,728.0000",$199.13,11.0000,10.0000,NaN,NaN,2024-11-09,HYDERABAD,Desktop,data analitics online
82,A1082,Data Analytcis Course,149.0000,"3,587.0000",$217.53,23.0000,8.0000,NaN,NaN,17-11-2024,hydrebad,Mobile,data analitics online
103,A1103,Data Analytcis Course,119.0000,"5,800.0000",$221.36,11.0000,8.0000,NaN,NaN,29-11-2024,hydrebad,Desktop,analytics for data
106,A1106,Data Analytics Corse,109.0000,"4,128.0000",$183.65,12.0000,5.0000,NaN,NaN,2024-11-30,HYDERABAD,Desktop,data analitics online
134,A1134,Data Analytcis Course,188.0000,"3,934.0000",$206.21,16.0000,9.0000,0.0480,NaN,04-11-2024,hydrebad,Desktop,data analytics course
137,A1137,Data Analytics Corse,122.0000,"4,048.0000",$202.23,28.0000,10.0000,0.0820,NaN,26-11-2024,hyderabad,mobile,online data analytic
160,A1160,Data Anlytics Corse,160.0000,NaN,$245.87,11.0000,6.0000,0.0370,NaN,2024-11-05,Hyderbad,DESKTOP,analytics for data
179,A1179,Data Analytcis Course,137.0000,"3,555.0000",$213.5,29.0000,3.0000,NaN,NaN,2024/11/06,hyderabad,DESKTOP,data analytics course



Conversions missing + Sale_Amount > 0
Records affected: 70


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
148,A1148,Data Analytics Corse,180.0000,"4,326.0000",$221.67,13.0000,NaN,NaN,$1427,06-11-2024,Hyderbad,TABLET,online data analytic
211,A1211,DataAnalyticsCourse,101.0000,"3,241.0000",$183.34,24.0000,NaN,NaN,$1483,10-11-2024,Hyderbad,TABLET,data anaytics training
213,A1213,Data Analytics Corse,199.0000,"5,734.0000",$247.99,18.0000,NaN,NaN,$1455,2024/11/11,hydrebad,DESKTOP,data analitics online
229,A1229,Data Anlytics Corse,165.0000,"4,551.0000",$185.35,30.0000,NaN,NaN,$1662,2024/11/06,Hyderbad,desktop,data analitics online
233,A1233,Data Anlytics Corse,161.0000,"5,977.0000",$193.27,24.0000,NaN,NaN,$1366,13-11-2024,Hyderbad,Desktop,online data analytic
304,A1304,Data Anlytics Corse,162.0000,"5,915.0000",$211.96,11.0000,NaN,NaN,$1394,2024-11-20,HYDERABAD,Tablet,analytics for data
345,A1345,Data Analytcis Course,NaN,"3,705.0000",$239.26,25.0000,NaN,NaN,$1506,02-11-2024,HYDERABAD,TABLET,online data analytic
459,A1459,Data Analytcis Course,128.0000,"4,532.0000",$183.28,11.0000,NaN,NaN,$1571,25-11-2024,hydrebad,Desktop,data anaytics training
520,A1520,Data Analytcis Course,127.0000,"5,964.0000",$182.01,26.0000,NaN,NaN,$1490,25-11-2024,Hyderbad,desktop,data analytics course
600,A1600,Data Anlytics Corse,142.0000,"5,999.0000",$214.05,27.0000,NaN,NaN,$1064,2024-11-21,HYDERABAD,desktop,analytics for data



Cost missing
Records affected: 97


...

,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
27,A1027,Data Analytics Corse,NaN,"5,286.0000",$234.89,22.0000,10.0000,NaN,$1688,2024-11-21,hyderabad,DESKTOP,data anaytics training
32,A1032,Data Analytcis Course,NaN,"3,872.0000",$244.38,20.0000,4.0000,NaN,$1041,12-11-2024,HYDERABAD,tablet,data analitics online
93,A1093,DataAnalyticsCourse,NaN,"5,903.0000",$184.41,23.0000,9.0000,NaN,$1188,20-11-2024,Hyderbad,tablet,online data analytic
141,A1141,Data Analytcis Course,NaN,"4,259.0000",$182.98,10.0000,10.0000,0.0310,$1048,16-11-2024,hyderabad,MOBILE,analytics for data
197,A1197,Data Analytcis Course,NaN,"5,016.0000",$238.59,26.0000,9.0000,NaN,$1620,2024-11-20,Hyderbad,Mobile,data analitics online
218,A1218,Data Anlytics Corse,NaN,"4,883.0000",$249.61,10.0000,3.0000,NaN,$1464,2024-11-12,hyderabad,Tablet,online data analytic
251,A1251,Data Analytics Corse,NaN,"4,629.0000",$236.96,10.0000,6.0000,NaN,$1183,06-11-2024,HYDERABAD,desktop,data anaytics training
289,A1289,Data Anlytics Corse,NaN,"5,277.0000",$227.46,10.0000,5.0000,NaN,$1541,2024/11/15,HYDERABAD,tablet,data analitics online
305,A1305,Data Analytcis Course,NaN,"4,450.0000",$219.21,16.0000,10.0000,0.0560,$1917,2024-11-10,hyderabad,mobile,learn data analytics
330,A1330,DataAnalyticsCourse,NaN,"4,615.0000",$194.89,12.0000,6.0000,NaN,$1923,2024/11/25,Hyderbad,TABLET,data anaytics training



Impressions missing
Records affected: 54


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
114,A1114,Data Analytics Corse,91.0000,NaN,$203.52,NaN,9.0000,0.0990,$1838,20-11-2024,hydrebad,DESKTOP,data analytics course
120,A1120,Data Anlytics Corse,117.0000,NaN,$214.18,29.0000,4.0000,0.0340,$1348,2024/11/23,hyderabad,MOBILE,learn data analytics
160,A1160,Data Anlytics Corse,160.0000,NaN,$245.87,11.0000,6.0000,0.0370,NaN,2024-11-05,Hyderbad,DESKTOP,analytics for data
210,A1210,Data Analytcis Course,199.0000,NaN,$199.98,15.0000,8.0000,0.0400,$1467,2024-11-08,HYDERABAD,tablet,data anaytics training
232,A1232,Data Anlytics Corse,101.0000,NaN,$234.17,29.0000,5.0000,0.0500,$1568,2024-11-27,hyderabad,mobile,data anaytics training
294,A1294,Data Anlytics Corse,90.0000,NaN,$249.0,21.0000,10.0000,NaN,$1245,19-11-2024,hyderabad,mobile,data analytics course
398,A1398,Data Analytcis Course,163.0000,NaN,$206.46,14.0000,3.0000,NaN,$1103,2024/11/12,HYDERABAD,tablet,learn data analytics
446,A1446,Data Analytcis Course,145.0000,NaN,$236.03,24.0000,8.0000,NaN,$1220,17-11-2024,hydrebad,Desktop,online data analytic
468,A1468,DataAnalyticsCourse,181.0000,NaN,$208.33,22.0000,8.0000,0.0440,$1932,2024-11-05,HYDERABAD,Tablet,online data analytic
471,A1471,Data Anlytics Corse,83.0000,NaN,$229.97,12.0000,8.0000,0.0960,$2000,30-11-2024,Hyderbad,DESKTOP,analytics for data



Leads missing
Records affected: 48


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
114,A1114,Data Analytics Corse,91.0000,NaN,$203.52,NaN,9.0000,0.0990,$1838,20-11-2024,hydrebad,DESKTOP,data analytics course
226,A1226,Data Anlytics Corse,91.0000,"4,290.0000",$213.74,NaN,10.0000,NaN,$1797,2024/11/16,hydrebad,Mobile,online data analytic
228,A1228,Data Analytics Corse,92.0000,"3,916.0000",$188.1,NaN,6.0000,0.0650,$1005,2024/11/14,hyderabad,DESKTOP,online data analytic
241,A1241,Data Analytcis Course,82.0000,"5,130.0000",$233.75,NaN,9.0000,0.1100,NaN,15-11-2024,Hyderbad,Mobile,data analitics online
298,A1298,Data Analytics Corse,95.0000,"3,397.0000",$181.89,NaN,7.0000,0.0740,$1560,2024/11/27,HYDERABAD,TABLET,data analytics course
537,A1537,Data Analytcis Course,196.0000,"3,777.0000",$246.52,NaN,5.0000,0.0260,$1159,2024-11-15,hyderabad,desktop,data analytics course
617,A1617,Data Analytcis Course,89.0000,"5,349.0000",NaN,NaN,8.0000,0.0900,$1371,28-11-2024,hydrebad,Tablet,data analitics online
675,A1675,Data Anlytics Corse,109.0000,"4,910.0000",$236.2,NaN,8.0000,0.0540,$1145,2024/11/08,hydrebad,TABLET,data anaytics training
733,A1733,DataAnalyticsCourse,190.0000,"3,567.0000",$249.26,NaN,7.0000,0.0570,$1517,2024/11/26,Hyderbad,Desktop,data analitics online
744,A1744,Data Analytics Corse,176.0000,"5,868.0000",$184.46,NaN,4.0000,0.0230,$1229,2024-11-20,HYDERABAD,desktop,data analytics course


In [35]:
# Relationship audit

relationship_pairs = {
    "Conversions ↔ Sale_Amount": (
        investigation["__Conversions"],
        investigation["__Sale_Amount"]
    ),
    "Conversions ↔ Leads": (
        investigation["__Conversions"],
        investigation["__Leads"]
    ),
    "Clicks ↔ Impressions": (
        investigation["__Clicks"],
        investigation["__Impressions"]
    ),
    "Cost ↔ Clicks": (
        investigation["__Cost"],
        investigation["__Clicks"]
    )
}

relationship_results = []

for relationship, (left, right) in relationship_pairs.items():
    complete = left.notna() & right.notna()
    
    relationship_results.append({
        "Relationship": relationship,
        "Complete_Pairs": int(complete.sum()),
        "Missing_One_or_Both": int((~complete).sum())
    })

relationship_audit = pd.DataFrame(relationship_results)

display(relationship_audit)

,Relationship,Complete_Pairs,Missing_One_or_Both
0,Conversions ↔ Sale_Amount,2391,209
1,Conversions ↔ Leads,2481,119
2,Clicks ↔ Impressions,2437,163
3,Cost ↔ Clicks,2397,203


In [36]:
# Preliminary classification of missing-value patterns

missing_value_meaning = pd.DataFrame({
    "Pattern": [
        "Sale_Amount missing + Conversions = 0",
        "Sale_Amount missing + Conversions > 0",
        "Conversions missing + Sale_Amount > 0",
        "Cost missing",
        "Clicks missing",
        "Impressions missing",
        "Leads missing"
    ],
    "Records_Affected": [
        int(sale_missing_conversions_zero.sum()),
        int(sale_missing_conversions_positive.sum()),
        int(conversions_missing_sale_positive.sum()),
        int(cost_missing.sum()),
        int(clicks_missing.sum()),
        int(impressions_missing.sum()),
        int(leads_missing.sum())
    ],
    "Preliminary_Classification": [
        "Requires business rule",
        "Potential data error / requires investigation",
        "Potential data error / requires investigation",
        "Unknown/missing — requires business rule",
        "Unknown/missing — requires business rule",
        "Unknown/missing — requires business rule",
        "Unknown/missing — requires business rule"
    ]
})

display(missing_value_meaning)

,Pattern,Records_Affected,Preliminary_Classification
0,Sale_Amount missing + Conversions = 0,0,Requires business rule
1,Sale_Amount missing + Conversions > 0,135,Potential data error / requires investigation
2,Conversions missing + Sale_Amount > 0,70,Potential data error / requires investigation
3,Cost missing,97,Unknown/missing — requires business rule
4,Clicks missing,112,Unknown/missing — requires business rule
5,Impressions missing,54,Unknown/missing — requires business rule
6,Leads missing,48,Unknown/missing — requires business rule


In [37]:
# Step 7 summary

print("MISSING-VALUE MEANING AUDIT")
print("-" * 50)

print(f"Total missing cells in raw dataset: {df.isna().sum().sum():,}")

print("\nImportant patterns:")
display(missing_value_meaning)

print("\nRelationships investigated:")
display(relationship_audit)

print("\nNo missing values were filled or removed.")
print("Raw DataFrame df remains unchanged.")

MISSING-VALUE MEANING AUDIT
--------------------------------------------------
Total missing cells in raw dataset: 1,150

Important patterns:


,Pattern,Records_Affected,Preliminary_Classification
0,Sale_Amount missing + Conversions = 0,0,Requires business rule
1,Sale_Amount missing + Conversions > 0,135,Potential data error / requires investigation
2,Conversions missing + Sale_Amount > 0,70,Potential data error / requires investigation
3,Cost missing,97,Unknown/missing — requires business rule
4,Clicks missing,112,Unknown/missing — requires business rule
5,Impressions missing,54,Unknown/missing — requires business rule
6,Leads missing,48,Unknown/missing — requires business rule



Relationships investigated:


,Relationship,Complete_Pairs,Missing_One_or_Both
0,Conversions ↔ Sale_Amount,2391,209
1,Conversions ↔ Leads,2481,119
2,Clicks ↔ Impressions,2437,163
3,Cost ↔ Clicks,2397,203



No missing values were filled or removed.
Raw DataFrame df remains unchanged.


## 8. Duplicate Audit

The purpose of this step is to determine whether the raw dataset contains:

1. Complete duplicate rows.
2. Duplicate `Ad_ID` values.

Complete duplicate rows are checked using `df.duplicated()`.

`Ad_ID` is expected to uniquely identify each advertising record, so duplicate `Ad_ID` values are also investigated separately.

**Important:** This is an audit only. No duplicate records will be deleted or modified in Phase 2.

In [38]:
# Step 8 — Audit complete duplicate rows

duplicate_row_mask = df.duplicated()

duplicate_row_count = int(duplicate_row_mask.sum())

print("COMPLETE DUPLICATE ROW AUDIT")
print("-" * 40)
print(f"Total rows: {len(df):,}")
print(f"Duplicate rows: {duplicate_row_count:,}")
print(f"Duplicate rows present: {duplicate_row_count > 0}")

if duplicate_row_count > 0:
    print("\nDuplicate rows found:")
    display(df.loc[duplicate_row_mask])
else:
    print("No complete duplicate rows found.")

COMPLETE DUPLICATE ROW AUDIT
----------------------------------------
Total rows: 2,600
Duplicate rows: 0
Duplicate rows present: False
No complete duplicate rows found.


In [39]:
# Audit duplicate Ad_ID values

ad_id_duplicate_mask = df["Ad_ID"].duplicated(keep=False)

duplicate_ad_id_count = int(df["Ad_ID"].duplicated().sum())
duplicate_ad_id_values = (
    df.loc[ad_id_duplicate_mask, "Ad_ID"]
    .dropna()
    .unique()
    .tolist()
)

print("AD_ID DUPLICATE AUDIT")
print("-" * 40)
print(f"Total Ad_ID records: {len(df):,}")
print(f"Duplicate Ad_ID records: {duplicate_ad_id_count:,}")
print(f"Unique duplicated Ad_ID values: {len(duplicate_ad_id_values):,}")

if duplicate_ad_id_values:
    print("\nDuplicated Ad_ID values:")
    print(duplicate_ad_id_values)
    display(
        df.loc[
            ad_id_duplicate_mask,
            ["Ad_ID", "Campaign_Name", "Ad_Date", "Location", "Device", "Keyword"]
        ]
    )
else:
    print("No duplicate Ad_ID values found.")

AD_ID DUPLICATE AUDIT
----------------------------------------
Total Ad_ID records: 2,600
Duplicate Ad_ID records: 0
Unique duplicated Ad_ID values: 0
No duplicate Ad_ID values found.


In [40]:
# Validate duplicate audit results

print("DUPLICATE AUDIT VALIDATION")
print("-" * 40)

print(f"Dataset rows: {len(df):,}")
print(f"Complete duplicate rows: {duplicate_row_count:,}")
print(f"Duplicate Ad_ID records: {duplicate_ad_id_count:,}")

print(
    f"\nComplete duplicate check passed: "
    f"{duplicate_row_count == 0}"
)

print(
    f"Ad_ID uniqueness check passed: "
    f"{duplicate_ad_id_count == 0}"
)

print("\nRaw DataFrame df modified: No")

DUPLICATE AUDIT VALIDATION
----------------------------------------
Dataset rows: 2,600
Complete duplicate rows: 0
Duplicate Ad_ID records: 0

Complete duplicate check passed: True
Ad_ID uniqueness check passed: True

Raw DataFrame df modified: No


In [41]:
# Concise Step 8 result for documentation

duplicate_audit_result = pd.DataFrame({
    "Audit": [
        "Complete duplicate rows",
        "Duplicate Ad_ID"
    ],
    "Records_Affected": [
        duplicate_row_count,
        duplicate_ad_id_count
    ],
    "Result": [
        "No duplicates" if duplicate_row_count == 0 else "Duplicates detected",
        "No duplicate Ad_IDs" if duplicate_ad_id_count == 0 else "Duplicate Ad_IDs detected"
    ],
    "Phase_2_Action": [
        "Document only" if duplicate_row_count == 0 else "Investigate; do not delete",
        "Document only" if duplicate_ad_id_count == 0 else "Investigate; do not modify"
    ]
})

display(duplicate_audit_result)

,Audit,Records_Affected,Result,Phase_2_Action
0,Complete duplicate rows,0,No duplicates,Document only
1,Duplicate Ad_ID,0,No duplicate Ad_IDs,Document only


## 9. Categorical Audit

The purpose of this step is to profile the categorical fields in the raw dataset and identify inconsistencies that may affect grouping, filtering, and campaign-performance analysis.

The following columns will be audited:

- `Campaign_Name`
- `Location`
- `Device`
- `Keyword`

For each column, the audit will examine:

- Unique values
- Frequency of each value
- Missing values
- Case inconsistencies
- Spelling inconsistencies
- Formatting inconsistencies

Known campaign, location, and device variants will be specifically checked.

**Important:** No categorical values will be standardized, merged, renamed, or otherwise modified in Phase 2. Identified issues will be documented for treatment in Phase 3.

In [42]:
# Step 9 — Categorical audit
# Read-only audit. The raw DataFrame df is not modified.

categorical_columns = [
    "Campaign_Name",
    "Location",
    "Device",
    "Keyword"
]

categorical_profile = pd.DataFrame({
    "Column": categorical_columns,
    "Unique_Count": [
        df[column].nunique(dropna=False)
        for column in categorical_columns
    ],
    "Non_Null_Count": [
        df[column].notna().sum()
        for column in categorical_columns
    ],
    "Missing_Count": [
        df[column].isna().sum()
        for column in categorical_columns
    ]
})

display(categorical_profile)

,Column,Unique_Count,Non_Null_Count,Missing_Count
0,Campaign_Name,4,2600,0
1,Location,4,2600,0
2,Device,9,2600,0
3,Keyword,6,2600,0


In [43]:
# Frequency distribution for every categorical column

for column in categorical_columns:
    print("\n" + "=" * 70)
    print(f"{column} — VALUE FREQUENCY")
    print("=" * 70)

    frequency_table = (
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="Frequency")
    )

    frequency_table["Percentage"] = (
        frequency_table["Frequency"] / len(df) * 100
    )

    display(frequency_table)


Campaign_Name — VALUE FREQUENCY


,Campaign_Name,Frequency,Percentage
0,Data Analytcis Course,680,26.1538
1,Data Analytics Corse,647,24.8846
2,DataAnalyticsCourse,637,24.5000
3,Data Anlytics Corse,636,24.4615



Location — VALUE FREQUENCY


,Location,Frequency,Percentage
0,HYDERABAD,661,25.4231
1,Hyderbad,656,25.2308
2,hyderabad,650,25.0000
3,hydrebad,633,24.3462



Device — VALUE FREQUENCY


,Device,Frequency,Percentage
0,MOBILE,311,11.9615
1,desktop,305,11.7308
2,Desktop,305,11.7308
3,tablet,305,11.7308
4,Mobile,291,11.1923
5,TABLET,279,10.7308
6,DESKTOP,278,10.6923
7,mobile,276,10.6154
8,Tablet,250,9.6154



Keyword — VALUE FREQUENCY


,Keyword,Frequency,Percentage
0,online data analytic,453,17.4231
1,learn data analytics,444,17.0769
2,data analytics course,440,16.9231
3,analytics for data,429,16.5000
4,data analitics online,420,16.1538
5,data anaytics training,414,15.9231


In [44]:
# Known Campaign_Name variants

known_campaign_variants = [
    "Data Analytics Course",
    "Data Analytcis Course",
    "DataAnalyticsCourse",
    "Data Anlytics Corse"
]

campaign_variant_audit = pd.DataFrame({
    "Raw_Value": known_campaign_variants,
    "Records_Affected": [
        int((df["Campaign_Name"] == value).sum())
        for value in known_campaign_variants
    ]
})

campaign_variant_audit["Present_In_Raw_Data"] = (
    campaign_variant_audit["Records_Affected"] > 0
)

display(campaign_variant_audit)

,Raw_Value,Records_Affected,Present_In_Raw_Data
0,Data Analytics Course,0,False
1,Data Analytcis Course,680,True
2,DataAnalyticsCourse,637,True
3,Data Anlytics Corse,636,True


In [45]:
# Inspect all Campaign_Name values exactly as stored in the raw dataset

campaign_values = (
    df["Campaign_Name"]
    .value_counts(dropna=False)
    .rename_axis("Campaign_Name")
    .reset_index(name="Frequency")
)

display(campaign_values)

print(
    f"Raw Campaign_Name unique values: "
    f"{df['Campaign_Name'].nunique(dropna=False)}"
)

print(
    f"Case-insensitive Campaign_Name values: "
    f"{df['Campaign_Name'].astype('string').str.casefold().nunique(dropna=True)}"
)

,Campaign_Name,Frequency
0,Data Analytcis Course,680
1,Data Analytics Corse,647
2,DataAnalyticsCourse,637
3,Data Anlytics Corse,636


Raw Campaign_Name unique values: 4
Case-insensitive Campaign_Name values: 4


In [46]:
# Known Location variants

known_location_variants = [
    "HYDERABAD",
    "Hyderbad",
    "hyderabad",
    "hydrebad"
]

location_variant_audit = pd.DataFrame({
    "Raw_Value": known_location_variants,
    "Records_Affected": [
        int((df["Location"] == value).sum())
        for value in known_location_variants
    ]
})

location_variant_audit["Present_In_Raw_Data"] = (
    location_variant_audit["Records_Affected"] > 0
)

display(location_variant_audit)

,Raw_Value,Records_Affected,Present_In_Raw_Data
0,HYDERABAD,661,True
1,Hyderbad,656,True
2,hyderabad,650,True
3,hydrebad,633,True


In [47]:
# Inspect all Location values exactly as stored

location_values = (
    df["Location"]
    .value_counts(dropna=False)
    .rename_axis("Location")
    .reset_index(name="Frequency")
)

display(location_values)

print(
    f"Raw Location unique values: "
    f"{df['Location'].nunique(dropna=False)}"
)

print(
    f"Case-insensitive Location values: "
    f"{df['Location'].astype('string').str.casefold().nunique(dropna=True)}"
)

,Location,Frequency
0,HYDERABAD,661
1,Hyderbad,656
2,hyderabad,650
3,hydrebad,633


Raw Location unique values: 4
Case-insensitive Location values: 3


In [48]:
# Known Device variants

known_device_variants = [
    "MOBILE",
    "Mobile",
    "mobile",
    "DESKTOP",
    "Desktop",
    "desktop",
    "TABLET",
    "Tablet",
    "tablet"
]

device_variant_audit = pd.DataFrame({
    "Raw_Value": known_device_variants,
    "Records_Affected": [
        int((df["Device"] == value).sum())
        for value in known_device_variants
    ]
})

device_variant_audit["Present_In_Raw_Data"] = (
    device_variant_audit["Records_Affected"] > 0
)

display(device_variant_audit)

,Raw_Value,Records_Affected,Present_In_Raw_Data
0,MOBILE,311,True
1,Mobile,291,True
2,mobile,276,True
3,DESKTOP,278,True
4,Desktop,305,True
5,desktop,305,True
6,TABLET,279,True
7,Tablet,250,True
8,tablet,305,True


In [49]:
# Inspect all Device values exactly as stored

device_values = (
    df["Device"]
    .value_counts(dropna=False)
    .rename_axis("Device")
    .reset_index(name="Frequency")
)

display(device_values)

print(
    f"Raw Device unique values: "
    f"{df['Device'].nunique(dropna=False)}"
)

print(
    f"Case-insensitive Device values: "
    f"{df['Device'].astype('string').str.casefold().nunique(dropna=True)}"
)

,Device,Frequency
0,MOBILE,311
1,desktop,305
2,Desktop,305
3,tablet,305
4,Mobile,291
5,TABLET,279
6,DESKTOP,278
7,mobile,276
8,Tablet,250


Raw Device unique values: 9
Case-insensitive Device values: 3


In [50]:
# Detect case/whitespace variants without modifying df

formatting_audit_results = []

for column in categorical_columns:
    raw_values = df[column].dropna().astype("string")
    
    raw_unique = raw_values.nunique()
    
    case_normalized_unique = raw_values.str.casefold().nunique()
    
    whitespace_normalized_unique = (
        raw_values.str.strip().nunique()
    )
    
    case_whitespace_normalized_unique = (
        raw_values.str.strip().str.casefold().nunique()
    )
    
    formatting_audit_results.append({
        "Column": column,
        "Raw_Unique": raw_unique,
        "Case_Normalized_Unique": case_normalized_unique,
        "Whitespace_Normalized_Unique": whitespace_normalized_unique,
        "Case_Whitespace_Normalized_Unique": case_whitespace_normalized_unique,
        "Potential_Case_Formatting_Variants": (
            raw_unique - case_whitespace_normalized_unique
        )
    })

formatting_audit = pd.DataFrame(formatting_audit_results)

display(formatting_audit)

,Column,Raw_Unique,Case_Normalized_Unique,Whitespace_Normalized_Unique,Case_Whitespace_Normalized_Unique,Potential_Case_Formatting_Variants
0,Campaign_Name,4,4,4,4,0
1,Location,4,3,4,3,1
2,Device,9,3,9,3,6
3,Keyword,6,6,6,6,0


In [51]:
# Show the exact raw values for each categorical column

for column in categorical_columns:
    print("\n" + "=" * 70)
    print(f"{column} — EXACT RAW VALUES")
    print("=" * 70)

    values = sorted(
        df[column]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for value in values:
        print(repr(value))


Campaign_Name — EXACT RAW VALUES
'Data Analytcis Course'
'Data Analytics Corse'
'Data Anlytics Corse'
'DataAnalyticsCourse'

Location — EXACT RAW VALUES
'HYDERABAD'
'Hyderbad'
'hyderabad'
'hydrebad'

Device — EXACT RAW VALUES
'DESKTOP'
'Desktop'
'MOBILE'
'Mobile'
'TABLET'
'Tablet'
'desktop'
'mobile'
'tablet'

Keyword — EXACT RAW VALUES
'analytics for data'
'data analitics online'
'data analytics course'
'data anaytics training'
'learn data analytics'
'online data analytic'


In [52]:
# Step 9 — Categorical audit summary

print("CATEGORICAL AUDIT SUMMARY")
print("=" * 50)

for column in categorical_columns:
    print(
        f"{column}: "
        f"{df[column].nunique(dropna=True)} unique values, "
        f"{df[column].isna().sum()} missing values"
    )

print("\nKnown Campaign variants:")
display(campaign_variant_audit)

print("\nKnown Location variants:")
display(location_variant_audit)

print("\nKnown Device variants:")
display(device_variant_audit)

print("\nFormatting audit:")
display(formatting_audit)

print("\nRaw DataFrame df modified: No")

CATEGORICAL AUDIT SUMMARY
Campaign_Name: 4 unique values, 0 missing values
Location: 4 unique values, 0 missing values
Device: 9 unique values, 0 missing values
Keyword: 6 unique values, 0 missing values

Known Campaign variants:


,Raw_Value,Records_Affected,Present_In_Raw_Data
0,Data Analytics Course,0,False
1,Data Analytcis Course,680,True
2,DataAnalyticsCourse,637,True
3,Data Anlytics Corse,636,True



Known Location variants:


,Raw_Value,Records_Affected,Present_In_Raw_Data
0,HYDERABAD,661,True
1,Hyderbad,656,True
2,hyderabad,650,True
3,hydrebad,633,True



Known Device variants:


,Raw_Value,Records_Affected,Present_In_Raw_Data
0,MOBILE,311,True
1,Mobile,291,True
2,mobile,276,True
3,DESKTOP,278,True
4,Desktop,305,True
5,desktop,305,True
6,TABLET,279,True
7,Tablet,250,True
8,tablet,305,True



Formatting audit:


,Column,Raw_Unique,Case_Normalized_Unique,Whitespace_Normalized_Unique,Case_Whitespace_Normalized_Unique,Potential_Case_Formatting_Variants
0,Campaign_Name,4,4,4,4,0
1,Location,4,3,4,3,1
2,Device,9,3,9,3,6
3,Keyword,6,6,6,6,0



Raw DataFrame df modified: No


## 10 Audit Keywords Separately

The purpose of this step is to audit the `Keyword` field independently from the other categorical variables.

The audit will:

- List every raw keyword value and its frequency.
- Check for case and formatting inconsistencies.
- Identify possible spelling variants.
- Classify potential issues without automatically merging or standardizing keywords.

Potential keyword issues will be classified as:

- **Clear spelling error**
- **Case/format issue**
- **Potentially different keyword**
- **Requires investigation**

**Important:** Keyword values will not be merged, corrected, or standardized during Phase 2. Any confirmed transformations will be performed only during Phase 3 after the business rule has been established.

In [53]:
# Step 10 — Audit keywords separately

keyword_frequency = (
    df["Keyword"]
    .value_counts(dropna=False)
    .rename_axis("Keyword")
    .reset_index(name="Frequency")
)

display(keyword_frequency)

,Keyword,Frequency
0,online data analytic,453
1,learn data analytics,444
2,data analytics course,440
3,analytics for data,429
4,data analitics online,420
5,data anaytics training,414


In [54]:
# Check whether different raw keyword values become identical
# after case normalization or whitespace normalization.

keyword_values = (
    df["Keyword"]
    .dropna()
    .astype(str)
)

keyword_audit = pd.DataFrame({
    "Raw_Keyword": keyword_values.unique()
})

keyword_audit["Case_Normalized"] = (
    keyword_audit["Raw_Keyword"]
    .str.casefold()
)

keyword_audit["Whitespace_Normalized"] = (
    keyword_audit["Raw_Keyword"]
    .str.strip()
    .str.casefold()
)

display(keyword_audit.sort_values("Raw_Keyword"))

,Raw_Keyword,Case_Normalized,Whitespace_Normalized
5,analytics for data,analytics for data,analytics for data
2,data analitics online,data analitics online,data analitics online
1,data analytics course,data analytics course,data analytics course
3,data anaytics training,data anaytics training,data anaytics training
0,learn data analytics,learn data analytics,learn data analytics
4,online data analytic,online data analytic,online data analytic


In [55]:
# Identify cases where multiple raw keyword values collapse
# to the same value after normalization.

keyword_variant_groups = (
    keyword_audit
    .groupby("Whitespace_Normalized")["Raw_Keyword"]
    .agg(list)
    .reset_index(name="Raw_Values")
)

keyword_variant_groups["Potential_Variant"] = (
    keyword_variant_groups["Raw_Values"].apply(len) > 1
)

display(
    keyword_variant_groups[
        keyword_variant_groups["Potential_Variant"]
    ]
)

,Whitespace_Normalized,Raw_Values,Potential_Variant


In [56]:
# Preliminary classification of keyword findings.
# No keyword values are changed or merged.

keyword_issue_audit = pd.DataFrame({
    "Keyword": sorted(df["Keyword"].dropna().unique()),
    "Classification": [
        "Potentially different keyword"
        for _ in sorted(df["Keyword"].dropna().unique())
    ]
})

display(keyword_issue_audit)

,Keyword,Classification
0,analytics for data,Potentially different keyword
1,data analitics online,Potentially different keyword
2,data analytics course,Potentially different keyword
3,data anaytics training,Potentially different keyword
4,learn data analytics,Potentially different keyword
5,online data analytic,Potentially different keyword


In [57]:
print("KEYWORD AUDIT VALIDATION")
print("-" * 40)

print(f"Unique raw keywords: {df['Keyword'].nunique(dropna=True)}")
print(f"Missing keywords: {df['Keyword'].isna().sum()}")
print(
    f"Keyword frequency total: "
    f"{keyword_frequency['Frequency'].sum():,}"
)

print(
    f"Frequency reconciles to dataset rows: "
    f"{keyword_frequency['Frequency'].sum() == len(df)}"
)

print("\nRaw DataFrame df modified: No")

KEYWORD AUDIT VALIDATION
----------------------------------------
Unique raw keywords: 6
Missing keywords: 0
Keyword frequency total: 2,600
Frequency reconciles to dataset rows: True

Raw DataFrame df modified: No


# Step 12 — Audit Dates

Inspect the raw `Ad_Date` values, identify the formats actually present, test explicit temporary parsing, and assess the expected business period of 2024-11-01 through 2024-11-30. The raw date column will not be converted.

In [58]:
# Capture the raw DataFrame fingerprint before any Step 12 temporary transformations.
original_df_shape = df.shape
original_df_columns = tuple(df.columns)
original_df_dtypes = tuple(str(dtype) for dtype in df.dtypes)
original_df_value_hash = pd.util.hash_pandas_object(df, index=True).values.tobytes()
print("Raw DataFrame fingerprint captured before Step 12 transformations.")

Raw DataFrame fingerprint captured before Step 12 transformations.


In [59]:
# Step 12 — Date inspection, explicit parsing, and range audit

raw_dates = df["Ad_Date"].astype("string")
date_missing_mask = df["Ad_Date"].isna()
date_non_null = int((~date_missing_mask).sum())

print("Raw Ad_Date inspection")
print(f"Raw dtype: {df['Ad_Date'].dtype}")
print(f"Missing: {int(date_missing_mask.sum())}")
print(f"Non-null: {date_non_null}")
print(f"Unique non-null values: {df['Ad_Date'].nunique(dropna=True)}")
print("Sample raw values:")
display(df[["Ad_ID", "Ad_Date"]].head(10))

# Classify formats before parsing so non-standard but valid dates remain distinct from invalid dates.
def classify_date_format(value):
    if pd.isna(value):
        return "Missing"
    text = str(value)
    if text != text.strip():
        return "Whitespace issue"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", text):
        return "YYYY-MM-DD"
    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", text):
        return "YYYY/MM/DD"
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", text):
        return "DD-MM-YYYY"
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", text):
        return "DD/MM/YYYY"
    return "Other/unrecognized"

def parse_date_explicit(value):
    if pd.isna(value):
        return pd.NaT
    text = str(value).strip()
    format_by_pattern = {
        "YYYY-MM-DD": "%Y-%m-%d",
        "YYYY/MM/DD": "%Y/%m/%d",
        "DD-MM-YYYY": "%d-%m-%Y",
        "DD/MM/YYYY": "%d/%m/%Y",
    }
    date_format = classify_date_format(text)
    if date_format not in format_by_pattern:
        return pd.NaT
    return pd.to_datetime(text, format=format_by_pattern[date_format], errors="coerce")

raw_date_formats = raw_dates.map(classify_date_format)
parsed_dates = raw_dates.map(parse_date_explicit)
expected_start = pd.Timestamp("2024-11-01")
expected_end = pd.Timestamp("2024-11-30")
parseable_mask = parsed_dates.notna()
unparseable_mask = (~date_missing_mask) & (~parseable_mask)
within_range_mask = parseable_mask & parsed_dates.between(expected_start, expected_end)
outside_range_mask = parseable_mask & (~parsed_dates.between(expected_start, expected_end))
ambiguous_mask = raw_dates.str.match(r"^\d{2}[/\-]\d{2}[/\-]\d{4}$", na=False) & parseable_mask
non_standard_mask = raw_date_formats.isin(["YYYY/MM/DD", "DD-MM-YYYY", "DD/MM/YYYY"])

format_counts = raw_date_formats.value_counts(dropna=False).rename_axis("Raw_Format").reset_index(name="Records")
format_counts["Percentage"] = format_counts["Records"] / len(df) * 100
print("Date formats actually present:")
display(format_counts)

date_audit_summary = pd.DataFrame({
    "Metric": [
        "Total records", "Missing", "Successfully parseable", "Unparseable",
        "Potentially ambiguous", "Non-standard but parseable", "Within expected range",
        "Outside expected range"
    ],
    "Records": [
        len(df), int(date_missing_mask.sum()), int(parseable_mask.sum()),
        int(unparseable_mask.sum()), int(ambiguous_mask.sum()),
        int((non_standard_mask & parseable_mask).sum()), int(within_range_mask.sum()),
        int(outside_range_mask.sum())
    ],
})
date_audit_summary["Percentage"] = date_audit_summary["Records"] / len(df) * 100
date_audit_summary["Finding"] = [
    "Raw dataset size", "Missing raw dates", "Valid under explicit format rules",
    "Non-missing values that did not parse", "DD-MM/DD-MM style values needing convention review",
    "Valid dates using a non-YYYY-MM-DD representation", "Dates in 2024-11-01 through 2024-11-30",
    "Parseable dates outside the expected business period"
]
print("Date audit summary:")
display(date_audit_summary)

date_investigation_records = df.loc[
    unparseable_mask | outside_range_mask | ambiguous_mask,
    ["Ad_ID", "Ad_Date"]
].copy()
date_investigation_records["Detected_Format"] = raw_date_formats.loc[date_investigation_records.index]
date_investigation_records["Parsed_Date_Temporary"] = parsed_dates.loc[date_investigation_records.index]
display(date_investigation_records.head(20))

assert date_audit_summary["Records"].iloc[0] == len(df)
assert date_audit_summary["Records"].iloc[1:].sum() >= 0
assert df["Ad_Date"].dtype == object
print("Raw Ad_Date permanently converted: No")

Raw Ad_Date inspection
Raw dtype: object
Missing: 0
Non-null: 2600
Unique non-null values: 90
Sample raw values:


,Ad_ID,Ad_Date
0,A1000,2024-11-16
1,A1001,20-11-2024
2,A1002,2024/11/16
3,A1003,2024-11-26
4,A1004,2024-11-22
5,A1005,16-11-2024
6,A1006,06-11-2024
7,A1007,2024/11/24
8,A1008,2024/11/24
9,A1009,2024/11/12


Date formats actually present:


,Raw_Format,Records,Percentage
0,YYYY-MM-DD,893,34.3462
1,DD-MM-YYYY,863,33.1923
2,YYYY/MM/DD,844,32.4615


Date audit summary:


,Metric,Records,Percentage,Finding
0,Total records,2600,100.0000,Raw dataset size
1,Missing,0,0.0000,Missing raw dates
2,Successfully parseable,2600,100.0000,Valid under explicit format rules
3,Unparseable,0,0.0000,Non-missing values that did not parse
4,Potentially ambiguous,863,33.1923,DD-MM/DD-MM style values needing convention re...
5,Non-standard but parseable,1707,65.6538,Valid dates using a non-YYYY-MM-DD representation
6,Within expected range,2600,100.0000,Dates in 2024-11-01 through 2024-11-30
7,Outside expected range,0,0.0000,Parseable dates outside the expected business ...


,Ad_ID,Ad_Date,Detected_Format,Parsed_Date_Temporary
1,A1001,20-11-2024,DD-MM-YYYY,2024-11-20
5,A1005,16-11-2024,DD-MM-YYYY,2024-11-16
6,A1006,06-11-2024,DD-MM-YYYY,2024-11-06
10,A1010,14-11-2024,DD-MM-YYYY,2024-11-14
13,A1013,12-11-2024,DD-MM-YYYY,2024-11-12
17,A1017,12-11-2024,DD-MM-YYYY,2024-11-12
23,A1023,30-11-2024,DD-MM-YYYY,2024-11-30
24,A1024,20-11-2024,DD-MM-YYYY,2024-11-20
31,A1031,22-11-2024,DD-MM-YYYY,2024-11-22
32,A1032,12-11-2024,DD-MM-YYYY,2024-11-12


Raw Ad_Date permanently converted: No


### Step 12 Findings

The raw date field contains multiple representations but all 2,600 records are explicitly parseable and fall within the expected November 2024 business period. The 1,707 non-standard representations require later standardization, while the 863 day-first records require confirmation of the source-system convention.

# Step 13 — Audit Currency Fields

Audit the raw `Cost` and `Sale_Amount` fields using temporary parsing only. Currency symbols and separators will be removed only in temporary Series for measurement; the raw columns remain unchanged.

In [60]:
# Step 13 — Currency audit with temporary parsing

def parse_currency_temporary(series):
    return pd.to_numeric(
        series.astype("string").str.replace(r"[$,]", "", regex=True).str.strip(),
        errors="coerce"
    )

currency_audit_rows = []
currency_problem_records = []
for currency_column in ["Cost", "Sale_Amount"]:
    raw_series = df[currency_column]
    parsed_series = parse_currency_temporary(raw_series)
    non_null_mask = raw_series.notna()
    parseable_mask_currency = parsed_series.notna() & non_null_mask
    non_parseable_mask_currency = non_null_mask & parsed_series.isna()
    raw_text = raw_series.astype("string")
    formatted_inconsistently = non_null_mask & ~raw_text.str.match(r"^\$[0-9,]+(?:\.[0-9]+)?$", na=False)
    currency_audit_rows.append({
        "Column": currency_column,
        "Raw_Data_Type": str(raw_series.dtype),
        "Non_Null_Count": int(non_null_mask.sum()),
        "Missing_Count": int(raw_series.isna().sum()),
        "Successfully_Parseable": int(parseable_mask_currency.sum()),
        "Non_Parseable": int(non_parseable_mask_currency.sum()),
        "Negative_Count": int((parsed_series < 0).sum()),
        "Zero_Count": int((parsed_series == 0).sum()),
        "Minimum_Parsed": parsed_series.min(),
        "Maximum_Parsed": parsed_series.max(),
        "Formatting_Review_Count": int(formatted_inconsistently.sum())
    })
    if non_parseable_mask_currency.any() or (parsed_series < 0).any():
        problem_rows = df.loc[non_parseable_mask_currency | (parsed_series < 0), ["Ad_ID", currency_column]].copy()
        problem_rows["Temporary_Parsed_Value"] = parsed_series.loc[problem_rows.index]
        currency_problem_records.append(problem_rows.assign(Column=currency_column))

currency_audit = pd.DataFrame(currency_audit_rows)
print("Currency audit summary:")
display(currency_audit)
print("Raw currency samples:")
display(df[["Ad_ID", "Cost", "Sale_Amount"]].head(10))
if currency_problem_records:
    currency_problem_records = pd.concat(currency_problem_records, ignore_index=True)
else:
    currency_problem_records = pd.DataFrame(columns=["Ad_ID", "Cost", "Sale_Amount", "Temporary_Parsed_Value", "Column"])
display(currency_problem_records)
assert currency_audit["Non_Null_Count"].add(currency_audit["Missing_Count"]).eq(len(df)).all()
assert df["Cost"].dtype == object and df["Sale_Amount"].dtype == object
print("Raw currency columns permanently converted: No")

Currency audit summary:


,Column,Raw_Data_Type,Non_Null_Count,Missing_Count,Successfully_Parseable,Non_Parseable,Negative_Count,Zero_Count,Minimum_Parsed,Maximum_Parsed,Formatting_Review_Count
0,Cost,object,2503,97,2503,0,0,0,180.0100,249.8900,0
1,Sale_Amount,object,2461,139,2461,0,0,0,"1,000.0000","2,000.0000",0


Raw currency samples:


,Ad_ID,Cost,Sale_Amount
0,A1000,$231.88,$1892
1,A1001,$216.84,$1679
2,A1002,$203.66,$1624
3,A1003,$237.66,$1225
4,A1004,$195.9,$1091
5,A1005,$243.57,$1315
6,A1006,$237.79,$1640
7,A1007,$229.61,$1509
8,A1008,NaN,$1362
9,A1009,$186.78,$1029


,Ad_ID,Cost,Sale_Amount,Temporary_Parsed_Value,Column


Raw currency columns permanently converted: No


### Step 13 Findings

The currency audit distinguishes missing values, parseable formatting variations, and genuinely non-parseable values. No parsed currency value is written back to `df`.

# Step 14 — Audit Numeric Fields

Audit raw numeric metrics and temporary numeric versions of currency fields. Count metrics are checked for unexpected decimals, and unusual values are reported without being labeled as errors automatically.

In [61]:
# Step 14 — Numeric audit

numeric_source_columns = ["Clicks", "Impressions", "Leads", "Conversions", "Conversion Rate", "Cost", "Sale_Amount"]
numeric_temp = {column: (parse_currency_temporary(df[column]) if column in ["Cost", "Sale_Amount"] else pd.to_numeric(df[column], errors="coerce")) for column in numeric_source_columns}
count_metric_columns = ["Clicks", "Impressions", "Leads", "Conversions"]
numeric_audit_rows = []
for numeric_column in numeric_source_columns:
    values = numeric_temp[numeric_column]
    raw_non_null = df[numeric_column].notna()
    integer_relevant = numeric_column in count_metric_columns
    numeric_audit_rows.append({
        "Column": numeric_column,
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Mean": values.mean(),
        "Median": values.median(),
        "Standard_Deviation": values.std(),
        "Missing_Count": int(df[numeric_column].isna().sum()),
        "Zero_Count": int((values == 0).sum()),
        "Negative_Count": int((values < 0).sum()),
        "Non_Parseable_Count": int(raw_non_null.sum() - values.notna().sum()),
        "Non_Integer_Count": int((~np.isclose(values.dropna() % 1, 0)).sum()) if integer_relevant else np.nan,
        "Values_Over_1": int((values > 1).sum()) if numeric_column == "Conversion Rate" else np.nan,
        "Values_Over_100": int((values > 100).sum()) if numeric_column == "Conversion Rate" else np.nan
    })
numeric_audit = pd.DataFrame(numeric_audit_rows)
print("Numeric audit summary:")
display(numeric_audit)

potentially_invalid_numeric_records = []
for numeric_column in numeric_source_columns:
    values = numeric_temp[numeric_column]
    invalid_mask = values < 0
    if numeric_column in count_metric_columns:
        invalid_mask = invalid_mask | (~np.isclose(values.fillna(0) % 1, 0) & values.notna())
    if numeric_column == "Conversion Rate":
        invalid_mask = invalid_mask | (values > 1)
    if invalid_mask.any():
        records = df.loc[invalid_mask, ["Ad_ID", numeric_column]].copy()
        records["Temporary_Parsed_Value"] = values.loc[records.index]
        records["Audit_Flag"] = "Potentially invalid or requires business-rule review"
        potentially_invalid_numeric_records.append(records.assign(Column=numeric_column))
if potentially_invalid_numeric_records:
    potentially_invalid_numeric_records = pd.concat(potentially_invalid_numeric_records, ignore_index=True)
else:
    potentially_invalid_numeric_records = pd.DataFrame(columns=["Ad_ID", "Column", "Temporary_Parsed_Value", "Audit_Flag"])
print("Potentially invalid numeric records:")
display(potentially_invalid_numeric_records.head(50))
assert set(numeric_audit["Column"]) == set(numeric_source_columns)
print("Raw numeric and currency columns modified: No")

Numeric audit summary:


,Column,Minimum,Maximum,Mean,Median,Standard_Deviation,Missing_Count,Zero_Count,Negative_Count,Non_Parseable_Count,Non_Integer_Count,Values_Over_1,Values_Over_100
0,Clicks,80.0000,199.0000,138.9570,139.0000,34.6194,112,0,0,0,0.0000,NaN,NaN
1,Impressions,"3,000.0000","5,999.0000","4,523.2808","4,518.5000",869.9279,54,0,0,0,0.0000,NaN,NaN
2,Leads,10.0000,30.0000,20.0039,20.0000,6.0323,48,0,0,0,0.0000,NaN,NaN
3,Conversions,3.0000,10.0000,6.5190,7.0000,2.2726,74,0,0,0,0.0000,NaN,NaN
4,Conversion Rate,0.0150,0.1230,0.0490,0.0460,0.0200,626,0,0,0,NaN,0.0000,0.0000
5,Cost,180.0100,249.8900,215.0906,215.5700,20.2896,97,0,0,0,NaN,NaN,NaN
6,Sale_Amount,"1,000.0000","2,000.0000","1,498.6481","1,505.0000",287.1066,139,0,0,0,NaN,NaN,NaN


Potentially invalid numeric records:


,Ad_ID,Column,Temporary_Parsed_Value,Audit_Flag


Raw numeric and currency columns modified: No


### Step 14 Findings

Numeric and currency fields are audited through temporary parsed values. Decimal behavior, negative values, and conversion-rate scale are surfaced for review, while no raw values are replaced.

# Step 15 — Business Rule Checks

Test documented relationships only where the required fields are present. Missing values are excluded from rule evaluation, and a violation is reported as an anomaly requiring interpretation rather than automatically called an error.

In [62]:
# Step 15 — Business-rule audit

rule_definitions = [
    ("Clicks <= Impressions", "constraint", "Clicks", "Impressions", lambda left, right: left <= right, "Supported constraint; inspect campaign tracking if violated."),
    ("Leads <= Clicks", "constraint", "Leads", "Clicks", lambda left, right: left <= right, "Potential business anomaly; confirm lead definition."),
    ("Conversions <= Clicks", "constraint", "Conversions", "Clicks", lambda left, right: left <= right, "Supported constraint if conversions are click-attributed."),
    ("Cost >= 0", "constraint", "Cost", None, lambda left, right: left >= 0, "Supported non-negative monetary constraint; confirm refunds/credits policy."),
    ("Sale_Amount >= 0", "constraint", "Sale_Amount", None, lambda left, right: left >= 0, "Potential business anomaly if refunds or adjustments can be negative."),
    ("Conversion_Rate >= 0", "constraint", "Conversion Rate", None, lambda left, right: left >= 0, "Supported non-negative rate constraint."),
    ("Conversions > Leads", "anomaly", "Conversions", "Leads", lambda left, right: left > right, "Requires business-definition confirmation; conversion and lead stages may differ."),
    ("Conversions > Clicks", "anomaly", "Conversions", "Clicks", lambda left, right: left > right, "Potential data error if one conversion cannot exceed one click."),
    ("Clicks > Impressions", "anomaly", "Clicks", "Impressions", lambda left, right: left > right, "Potential data error; inspect source metric definitions."),
]

business_rule_rows = []
business_rule_violation_records = []
for rule_name, rule_type, left_column, right_column, rule_function, interpretation in rule_definitions:
    left_values = numeric_temp[left_column]
    right_values = numeric_temp[right_column] if right_column else None
    evaluable = left_values.notna() & (right_values.notna() if right_values is not None else True)
    condition_result = pd.Series(False, index=df.index)
    condition_result.loc[evaluable] = rule_function(left_values.loc[evaluable], right_values.loc[evaluable] if right_values is not None else None)
    if rule_type == "constraint":
        violations = evaluable & ~condition_result
        records_satisfying = int((evaluable & condition_result).sum())
        records_violating = int(violations.sum())
        records_flagged = records_violating
    else:
        violations = pd.Series(False, index=df.index)
        records_satisfying = int((evaluable & condition_result).sum())
        records_violating = 0
        records_flagged = records_satisfying
    business_rule_rows.append({
        "Rule": rule_name,
        "Rule_Type": rule_type,
        "Records_Evaluated": int(evaluable.sum()),
        "Records_Satisfying": records_satisfying,
        "Records_Violating": records_violating,
        "Records_Flagged": records_flagged,
        "Percentage_Violating": float(records_violating / evaluable.sum() * 100) if evaluable.any() else 0.0,
        "Interpretation": interpretation,
        "Recommended_Investigation": "Inspect affected raw records and confirm source-system definition before Phase 3 treatment."
    })
    if records_flagged:
        flagged_records = df.loc[evaluable & (condition_result if rule_type == "anomaly" else ~condition_result), ["Ad_ID", left_column] + ([right_column] if right_column else [])].copy()
        flagged_records["Rule"] = rule_name
        flagged_records["Rule_Type"] = rule_type
        business_rule_violation_records.append(flagged_records)

business_rule_audit = pd.DataFrame(business_rule_rows)
if business_rule_violation_records:
    business_rule_violation_records = pd.concat(business_rule_violation_records, ignore_index=True)
else:
    business_rule_violation_records = pd.DataFrame(columns=["Ad_ID", "Rule", "Rule_Type"])
print("Business-rule audit:")
display(business_rule_audit)
print("Constraint violations and anomaly flags:")
display(business_rule_violation_records.head(50))
assert (business_rule_audit["Records_Evaluated"] <= len(df)).all()
print("Missing values treated as violations: No")

Business-rule audit:


,Rule,Rule_Type,Records_Evaluated,Records_Satisfying,Records_Violating,Records_Flagged,Percentage_Violating,Interpretation,Recommended_Investigation
0,Clicks <= Impressions,constraint,2437,2437,0,0,0.0000,Supported constraint; inspect campaign trackin...,Inspect affected raw records and confirm sourc...
1,Leads <= Clicks,constraint,2443,2443,0,0,0.0000,Potential business anomaly; confirm lead defin...,Inspect affected raw records and confirm sourc...
2,Conversions <= Clicks,constraint,2417,2417,0,0,0.0000,Supported constraint if conversions are click-...,Inspect affected raw records and confirm sourc...
3,Cost >= 0,constraint,2503,2503,0,0,0.0000,Supported non-negative monetary constraint; co...,Inspect affected raw records and confirm sourc...
4,Sale_Amount >= 0,constraint,2461,2461,0,0,0.0000,Potential business anomaly if refunds or adjus...,Inspect affected raw records and confirm sourc...
5,Conversion_Rate >= 0,constraint,1974,1974,0,0,0.0000,Supported non-negative rate constraint.,Inspect affected raw records and confirm sourc...
6,Conversions > Leads,anomaly,2481,0,0,0,0.0000,Requires business-definition confirmation; con...,Inspect affected raw records and confirm sourc...
7,Conversions > Clicks,anomaly,2417,0,0,0,0.0000,Potential data error if one conversion cannot ...,Inspect affected raw records and confirm sourc...
8,Clicks > Impressions,anomaly,2437,0,0,0,0.0000,Potential data error; inspect source metric de...,Inspect affected raw records and confirm sourc...


Constraint violations and anomaly flags:


,Ad_ID,Rule,Rule_Type


Missing values treated as violations: No


### Step 15 Findings

Each relationship is evaluated only on complete records. Any violations remain investigative findings because source-system definitions, attribution rules, refunds, or stage semantics can change their interpretation.

# Step 16 — Audit Existing Conversion Rate

Audit the raw `Conversion Rate` field as stored, determine whether it is a decimal proportion, and compare it temporarily with `Conversions / Clicks`. The raw field will not be overwritten or recomputed.

In [63]:
# Step 16 — Existing conversion-rate audit

raw_conversion_rate = pd.to_numeric(df["Conversion Rate"], errors="coerce")
conversion_rate_audit = pd.DataFrame([{
    "Missing_Count": int(raw_conversion_rate.isna().sum()),
    "Minimum": raw_conversion_rate.min(),
    "Maximum": raw_conversion_rate.max(),
    "Mean": raw_conversion_rate.mean(),
    "Median": raw_conversion_rate.median(),
    "Zero_Count": int((raw_conversion_rate == 0).sum()),
    "Negative_Count": int((raw_conversion_rate < 0).sum()),
    "Values_Over_1": int((raw_conversion_rate > 1).sum()),
    "Values_Over_100": int((raw_conversion_rate > 100).sum()),
    "Outside_Expected_0_to_1": int(((raw_conversion_rate < 0) | (raw_conversion_rate > 1)).sum())
}])

comparison_mask = (
    numeric_temp["Clicks"].notna()
    & numeric_temp["Conversions"].notna()
    & numeric_temp["Clicks"].ne(0)
    & raw_conversion_rate.notna()
)
calculated_conversion_rate = pd.Series(np.nan, index=df.index, dtype="float64")
calculated_conversion_rate.loc[comparison_mask] = (
    numeric_temp["Conversions"].loc[comparison_mask] / numeric_temp["Clicks"].loc[comparison_mask]
)
conversion_rate_comparison = df.loc[comparison_mask, ["Ad_ID", "Clicks", "Conversions", "Conversion Rate"]].copy()
conversion_rate_comparison["calculated_conversion_rate"] = calculated_conversion_rate.loc[comparison_mask]
conversion_rate_comparison["absolute_difference"] = (
    raw_conversion_rate.loc[comparison_mask] - calculated_conversion_rate.loc[comparison_mask]
).abs()
conversion_rate_comparison["relative_difference"] = (
    conversion_rate_comparison["absolute_difference"]
    / calculated_conversion_rate.loc[comparison_mask].abs().replace(0, np.nan)
)
matching_mask = conversion_rate_comparison["absolute_difference"].le(0.0005)
materially_inconsistent_mask = conversion_rate_comparison["absolute_difference"].gt(0.01)
conversion_rate_audit["Comparable_Records"] = int(comparison_mask.sum())
conversion_rate_audit["Matching_Records"] = int(matching_mask.sum())
conversion_rate_audit["Materially_Inconsistent_Records"] = int(materially_inconsistent_mask.sum())
conversion_rate_audit["Representation_Interpretation"] = "Decimal proportion if values are generally between 0 and 1"
print("Conversion Rate audit:")
display(conversion_rate_audit)
print("Raw versus temporary calculated rate sample:")
display(conversion_rate_comparison.head(20))
print("Materially inconsistent comparisons:")
display(conversion_rate_comparison.loc[materially_inconsistent_mask].head(20))
assert df["Conversion Rate"].dtype == float
print("Conversion Rate recomputed or overwritten: No")

Conversion Rate audit:


,Missing_Count,Minimum,Maximum,Mean,Median,Zero_Count,Negative_Count,Values_Over_1,Values_Over_100,Outside_Expected_0_to_1,Comparable_Records,Matching_Records,Materially_Inconsistent_Records,Representation_Interpretation
0,626,0.0150,0.1230,0.0490,0.0460,0,0,0,0,0,1944,1430,331,Decimal proportion if values are generally bet...


Raw versus temporary calculated rate sample:


,Ad_ID,Clicks,Conversions,Conversion Rate,calculated_conversion_rate,absolute_difference,relative_difference
0,A1000,104.0000,7.0000,0.0580,0.0673,0.0093,0.1383
1,A1001,173.0000,8.0000,0.0460,0.0462,0.0002,0.0052
6,A1006,116.0000,5.0000,0.0430,0.0431,0.0001,0.0024
7,A1007,184.0000,3.0000,0.0160,0.0163,0.0003,0.0187
8,A1008,113.0000,4.0000,0.0580,0.0354,0.0226,0.6385
9,A1009,166.0000,9.0000,0.0540,0.0542,0.0002,0.0040
10,A1010,101.0000,6.0000,0.0590,0.0594,0.0004,0.0068
11,A1011,101.0000,5.0000,0.0500,0.0495,0.0005,0.0100
12,A1012,125.0000,3.0000,0.0240,0.0240,0.0000,0.0000
13,A1013,196.0000,7.0000,0.0380,0.0357,0.0023,0.0640


Materially inconsistent comparisons:


,Ad_ID,Clicks,Conversions,Conversion Rate,calculated_conversion_rate,absolute_difference,relative_difference
8,A1008,113.0000,4.0000,0.0580,0.0354,0.0226,0.6385
20,A1020,184.0000,10.0000,0.0440,0.0543,0.0103,0.1904
23,A1023,188.0000,5.0000,0.0470,0.0266,0.0204,0.7672
26,A1026,119.0000,10.0000,0.0600,0.0840,0.0240,0.2860
33,A1033,176.0000,7.0000,0.0540,0.0398,0.0142,0.3577
35,A1035,104.0000,9.0000,0.0310,0.0865,0.0555,0.6418
36,A1036,146.0000,8.0000,0.0350,0.0548,0.0198,0.3612
38,A1038,90.0000,6.0000,0.0430,0.0667,0.0237,0.3550
46,A1046,188.0000,4.0000,0.0480,0.0213,0.0267,1.2560
50,A1050,192.0000,5.0000,0.0390,0.0260,0.0130,0.4976


Conversion Rate recomputed or overwritten: No


### Step 16 Findings

The comparison determines whether the stored rate behaves like a decimal proportion and separates missingness from material disagreement. Any recomputation remains a Phase 5 activity, as required by the project plan.

# Step 17 — Detect Outliers

Use the IQR method on temporary numeric values for the requested metrics. Outliers are reported for investigation only and are never deleted, capped, or replaced.

In [64]:
# Step 17 — IQR outlier audit

outlier_columns = ["Impressions", "Clicks", "Cost", "Leads", "Conversions", "Sale_Amount"]
outlier_summary_rows = []
outlier_record_frames = []
for outlier_column in outlier_columns:
    values = numeric_temp[outlier_column]
    non_missing = values.dropna()
    q1 = non_missing.quantile(0.25)
    q3 = non_missing.quantile(0.75)
    iqr = q3 - q1
    lower_boundary = q1 - 1.5 * iqr
    upper_boundary = q3 + 1.5 * iqr
    lower_mask = values < lower_boundary
    upper_mask = values > upper_boundary
    outlier_mask = lower_mask | upper_mask
    outlier_summary_rows.append({
        "Column": outlier_column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Boundary": lower_boundary,
        "Upper_Boundary": upper_boundary,
        "Lower_Outliers": int(lower_mask.sum()),
        "Upper_Outliers": int(upper_mask.sum()),
        "Total_Outliers": int(outlier_mask.sum()),
        "Outlier_Percentage": float(outlier_mask.sum() / values.notna().sum() * 100) if values.notna().any() else 0.0
    })
    if outlier_mask.any():
        records = df.loc[outlier_mask, ["Ad_ID"]].copy()
        records["Column"] = outlier_column
        records["Value"] = values.loc[outlier_mask]
        records["Lower_Boundary"] = lower_boundary
        records["Upper_Boundary"] = upper_boundary
        records["Outlier_Direction"] = np.where(lower_mask.loc[outlier_mask], "Lower", "Upper")
        records["Classification"] = "Requires investigation; IQR detection is not proof of bad data"
        outlier_record_frames.append(records)

outlier_summary = pd.DataFrame(outlier_summary_rows)
outlier_records = pd.concat(outlier_record_frames, ignore_index=True) if outlier_record_frames else pd.DataFrame(columns=["Ad_ID", "Column", "Value", "Lower_Boundary", "Upper_Boundary", "Outlier_Direction", "Classification"])
print("Outlier summary:")
display(outlier_summary)
print("Outlier records sample:")
display(outlier_records.head(50))
assert set(outlier_summary["Column"]) == set(outlier_columns)
print("Outliers modified or deleted: No")

Outlier summary:


,Column,Q1,Q3,IQR,Lower_Boundary,Upper_Boundary,Lower_Outliers,Upper_Outliers,Total_Outliers,Outlier_Percentage
0,Impressions,"3,764.0000","5,279.5000","1,515.5000","1,490.7500","7,552.7500",0,0,0,0.0000
1,Clicks,110.0000,169.0000,59.0000,21.5000,257.5000,0,0,0,0.0000
2,Cost,197.5400,232.9900,35.4500,144.3650,286.1650,0,0,0,0.0000
3,Leads,15.0000,25.0000,10.0000,0.0000,40.0000,0,0,0,0.0000
4,Conversions,5.0000,9.0000,4.0000,-1.0000,15.0000,0,0,0,0.0000
5,Sale_Amount,"1,248.0000","1,742.0000",494.0000,507.0000,"2,483.0000",0,0,0,0.0000


Outlier records sample:


,Ad_ID,Column,Value,Lower_Boundary,Upper_Boundary,Outlier_Direction,Classification


Outliers modified or deleted: No


### Step 17 Findings

The IQR tables identify distributional extremes and provide actual records for review. Whether an extreme is a data error, legitimate performance, or a business anomaly must be decided from source and business context.

# Step 18 — Data Quality Issue Log

Create a structured issue log from the actual audit results in Steps 1–17. The log documents confirmed findings, potential issues, and business-rule dependencies for later phases without applying any cleaning.

In [65]:
# Step 18 — Build the issue log from calculated audit results

issue_rows = []
def add_issue(column, issue_type, description, records, severity, impact, treatment, phase):
    issue_rows.append({
        "Issue ID": f"DQ-{len(issue_rows) + 1:03d}",
        "Column": column,
        "Issue Type": issue_type,
        "Description": description,
        "Records Affected": int(records),
        "Percentage": float(records / len(df) * 100),
        "Severity": severity,
        "Business Impact": impact,
        "Proposed Treatment": treatment,
        "Phase": phase
    })

add_issue("Conversion Rate", "Column naming", "Raw name differs from planned analytical name: Conversion Rate versus Conversion_Rate.", int((df.columns == "Conversion Rate").sum()), "Low", "Can cause inconsistent references across analysis layers.", "Rename only in the approved cleaning phase.", "Phase 3")
for column in ["Conversion Rate", "Sale_Amount", "Clicks", "Cost", "Conversions", "Impressions", "Leads"]:
    missing_count = int(df[column].isna().sum())
    if missing_count:
        add_issue(column, "Missing values", f"{missing_count:,} raw records are missing ({missing_count / len(df) * 100:.2f}%).", missing_count, "Medium", "Reduces completeness of metric-based analysis.", "Apply an approved field-specific business rule; do not impute automatically.", "Phase 3")
add_issue("Sale_Amount / Conversions", "Missing-value relationship", "Sale_Amount is missing while Conversions is positive.", int((df["Sale_Amount"].isna() & numeric_temp["Conversions"].gt(0)).sum()), "High", "May understate revenue associated with recorded conversions.", "Investigate source-system capture and define treatment.", "Phase 3")
add_issue("Conversions / Sale_Amount", "Missing-value relationship", "Conversions is missing while Sale_Amount is positive.", int((df["Conversions"].isna() & numeric_temp["Sale_Amount"].gt(0)).sum()), "High", "May break conversion and revenue reconciliation.", "Investigate source-system capture and define treatment.", "Phase 3")
add_issue("Ad_Date", "Date formatting", "All dates parse explicitly, but 1,707 use non-YYYY-MM-DD representations; 863 use DD-MM-YYYY.", int((non_standard_mask & parseable_mask).sum()), "Medium", "Mixed representations can cause inconsistent grouping or parsing.", "Standardize representation after confirming day-first convention.", "Phase 3")
add_issue("Location", "Categorical variant", "Four raw values collapse to three after case normalization, including spelling variants.", int(df["Location"].nunique(dropna=True)), "Medium", "Can split location-level performance results.", "Review and standardize only under an approved mapping.", "Phase 3")
add_issue("Device", "Categorical variant", "Nine raw values collapse to three after case normalization, reflecting case variants.", int(df["Device"].nunique(dropna=True)), "Medium", "Can split device-level performance results.", "Review and standardize case in Phase 3.", "Phase 3")
add_issue("Campaign_Name", "Spelling/format variant", "Four distinct campaign strings are present, including spelling and spacing variants.", int(df["Campaign_Name"].nunique(dropna=True)), "Medium", "Can split campaign performance results.", "Confirm canonical campaign mapping before standardization.", "Phase 3")
add_issue("Keyword", "Keyword finding", "Six distinct raw keywords were observed; no values were merged automatically.", int(df["Keyword"].nunique(dropna=True)), "Low", "Keyword grouping may require business interpretation.", "Review search-term semantics before any mapping.", "Phase 3")
for _, row in outlier_summary[outlier_summary["Total_Outliers"] > 0].iterrows():
    add_issue(row["Column"], "Outlier", f"IQR flagged {int(row['Total_Outliers'])} records ({row['Outlier_Percentage']:.2f}%) for investigation.", int(row["Total_Outliers"]), "Low", "Extreme values can influence summary metrics and model behavior.", "Inspect source records and retain legitimate extremes.", "Phase 3")
for _, row in business_rule_audit[business_rule_audit["Records_Violating"] > 0].iterrows():
    add_issue(row["Rule"], "Business-rule violation", f"{int(row['Records_Violating'])} complete records violate the stated constraint.", int(row["Records_Violating"]), "High", "May affect metric validity if the rule is confirmed as applicable.", "Confirm source definitions before treatment.", "Phase 3")
for _, row in business_rule_audit[business_rule_audit["Records_Flagged"] > 0].iterrows():
    add_issue(row["Rule"], "Business anomaly", f"{int(row['Records_Flagged'])} complete records meet an anomaly-detection condition.", int(row["Records_Flagged"]), "Medium", "May indicate a source-definition or tracking issue.", "Confirm business definitions before treatment.", "Phase 3")
if int(conversion_rate_audit["Materially_Inconsistent_Records"].iloc[0]) > 0:
    add_issue("Conversion Rate", "Metric inconsistency", "Stored rates materially differ from temporary Conversions / Clicks calculations.", int(conversion_rate_audit["Materially_Inconsistent_Records"].iloc[0]), "High", "Can affect conversion-rate reporting.", "Recompute only under the approved Phase 5 rule.", "Phase 5")

issue_log = pd.DataFrame(issue_rows)
issue_log_path = Path("reports/profiling/data_quality_issue_log.xlsx")
issue_log_path.parent.mkdir(parents=True, exist_ok=True)
print(f"Existing issue log before update: {issue_log_path.exists()}")
issue_log.to_excel(issue_log_path, index=False, sheet_name="Issue_Log")
print(f"Issue log rows: {len(issue_log)}")
display(issue_log)
assert issue_log["Issue ID"].tolist() == [f"DQ-{i:03d}" for i in range(1, len(issue_log) + 1)]

Existing issue log before update: True
Issue log rows: 16


,Issue ID,Column,Issue Type,Description,Records Affected,Percentage,Severity,Business Impact,Proposed Treatment,Phase
0,DQ-001,Conversion Rate,Column naming,Raw name differs from planned analytical name:...,1,0.0385,Low,Can cause inconsistent references across analy...,Rename only in the approved cleaning phase.,Phase 3
1,DQ-002,Conversion Rate,Missing values,626 raw records are missing (24.08%).,626,24.0769,Medium,Reduces completeness of metric-based analysis.,Apply an approved field-specific business rule...,Phase 3
2,DQ-003,Sale_Amount,Missing values,139 raw records are missing (5.35%).,139,5.3462,Medium,Reduces completeness of metric-based analysis.,Apply an approved field-specific business rule...,Phase 3
3,DQ-004,Clicks,Missing values,112 raw records are missing (4.31%).,112,4.3077,Medium,Reduces completeness of metric-based analysis.,Apply an approved field-specific business rule...,Phase 3
4,DQ-005,Cost,Missing values,97 raw records are missing (3.73%).,97,3.7308,Medium,Reduces completeness of metric-based analysis.,Apply an approved field-specific business rule...,Phase 3
5,DQ-006,Conversions,Missing values,74 raw records are missing (2.85%).,74,2.8462,Medium,Reduces completeness of metric-based analysis.,Apply an approved field-specific business rule...,Phase 3
6,DQ-007,Impressions,Missing values,54 raw records are missing (2.08%).,54,2.0769,Medium,Reduces completeness of metric-based analysis.,Apply an approved field-specific business rule...,Phase 3
7,DQ-008,Leads,Missing values,48 raw records are missing (1.85%).,48,1.8462,Medium,Reduces completeness of metric-based analysis.,Apply an approved field-specific business rule...,Phase 3
8,DQ-009,Sale_Amount / Conversions,Missing-value relationship,Sale_Amount is missing while Conversions is po...,135,5.1923,High,May understate revenue associated with recorde...,Investigate source-system capture and define t...,Phase 3
9,DQ-010,Conversions / Sale_Amount,Missing-value relationship,Conversions is missing while Sale_Amount is po...,70,2.6923,High,May break conversion and revenue reconciliation.,Investigate source-system capture and define t...,Phase 3


# Step 19 — Complete Excel Audit

Update the existing workbook without destroying unrelated sheets. Replace only the named audit sheets with results calculated in this notebook; analytical performance tables are not created in this step.

In [66]:
# Step 19 — Write calculated audit tables to the existing workbook

from openpyxl import load_workbook
from zipfile import BadZipFile

project_root = raw_path.parents[2]
excel_path = project_root / "excel" / "marketing_campaign_analysis.xlsx"
issue_log_path = project_root / "reports" / "profiling" / "data_quality_issue_log.xlsx"
issue_log_path.parent.mkdir(parents=True, exist_ok=True)
profile_export = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isna().sum().values,
    "Missing_%": (df.isna().mean() * 100).values,
    "Unique_Count": df.nunique(dropna=True).values
})
missing_export = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isna().sum().values,
    "Missing_%": df.isna().mean().values * 100,
    "Non_Missing_Count": df.notna().sum().values
}).sort_values("Missing_Count", ascending=False).reset_index(drop=True)
duplicate_export = pd.DataFrame({
    "Audit": ["Complete duplicate rows", "Duplicate Ad_ID"],
    "Records_Affected": [int(df.duplicated().sum()), int(df["Ad_ID"].duplicated().sum())],
    "Result": ["No duplicates" if not df.duplicated().any() else "Duplicates detected", "No duplicate Ad_IDs" if not df["Ad_ID"].duplicated().any() else "Duplicate Ad_IDs detected"]
})
categorical_export = pd.DataFrame({
    "Column": ["Campaign_Name", "Location", "Device", "Keyword"],
    "Unique_Count": [df[column].nunique(dropna=False) for column in ["Campaign_Name", "Location", "Device", "Keyword"]],
    "Missing_Count": [int(df[column].isna().sum()) for column in ["Campaign_Name", "Location", "Device", "Keyword"]]
})

existing_workbook_sheets = []
if excel_path.exists():
    try:
        existing_workbook_sheets = load_workbook(excel_path, read_only=True).sheetnames
    except BadZipFile:
        backup_path = excel_path.with_name(f"{excel_path.stem}.invalid.xlsx")
        suffix_number = 2
        while backup_path.exists():
            backup_path = excel_path.with_name(f"{excel_path.stem}.invalid_{suffix_number}.xlsx")
            suffix_number += 1
        excel_path.rename(backup_path)
        print(f"Invalid workbook preserved at: {backup_path}")

if not excel_path.exists():
    excel_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        pd.DataFrame({"Status": ["Created by Phase 2 audit; prior invalid workbook preserved separately"]}).to_excel(writer, sheet_name="Audit_Readme", index=False)
    existing_workbook_sheets = ["Audit_Readme"]
print("Existing workbook sheets:", existing_workbook_sheets)
audit_sheet_names = ["Data_Profile", "Missing_Values", "Duplicates", "Categories", "Date_Audit", "Numeric_Audit", "Outliers", "Data_Quality_Issue_Log"]
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    profile_export.to_excel(writer, sheet_name="Data_Profile", index=False)
    missing_export.to_excel(writer, sheet_name="Missing_Values", index=False)
    duplicate_export.to_excel(writer, sheet_name="Duplicates", index=False)
    categorical_export.to_excel(writer, sheet_name="Categories", index=False, startrow=0)
    for offset, category_column in enumerate(["Campaign_Name", "Location", "Device", "Keyword"], start=1):
        values_export = df[category_column].value_counts(dropna=False).rename_axis(category_column).reset_index(name="Frequency")
        values_export.to_excel(writer, sheet_name="Categories", index=False, startrow=len(categorical_export) + 3 + (offset - 1) * (len(values_export) + 3))
    date_audit_summary.to_excel(writer, sheet_name="Date_Audit", index=False)
    numeric_audit.to_excel(writer, sheet_name="Numeric_Audit", index=False)
    outlier_summary.to_excel(writer, sheet_name="Outliers", index=False)
    issue_log.to_excel(writer, sheet_name="Data_Quality_Issue_Log", index=False)
issue_log.to_excel(issue_log_path, index=False, sheet_name="Issue_Log")
final_workbook_sheets = load_workbook(excel_path, read_only=True).sheetnames
print("Updated workbook sheets:", final_workbook_sheets)
assert all(sheet in final_workbook_sheets for sheet in audit_sheet_names)
display(pd.DataFrame({"Sheet": audit_sheet_names, "Present": [sheet in final_workbook_sheets for sheet in audit_sheet_names]}))

Existing workbook sheets: ['Audit_Readme', 'Data_Profile', 'Missing_Values', 'Duplicates', 'Categories', 'Date_Audit', 'Numeric_Audit', 'Outliers', 'Data_Quality_Issue_Log']
Updated workbook sheets: ['Audit_Readme', 'Data_Profile', 'Missing_Values', 'Duplicates', 'Categories', 'Date_Audit', 'Numeric_Audit', 'Outliers', 'Data_Quality_Issue_Log']


,Sheet,Present
0,Data_Profile,True
1,Missing_Values,True
2,Duplicates,True
3,Categories,True
4,Date_Audit,True
5,Numeric_Audit,True
6,Outliers,True
7,Data_Quality_Issue_Log,True


### Step 19 Findings

The existing workbook is preserved while the eight requested audit sheets are refreshed from notebook DataFrames. No analytical performance tables are added.

# Step 20 — Create Profiling Report

Create the Phase 2 profiling report with the required structure and actual calculated results. The report distinguishes confirmed data-quality findings, potential issues, and business-definition dependencies.

In [67]:
# Step 20 — Write the profiling report and existing profiling text summaries

project_root = raw_path.parents[2]
report_path = project_root / "reports" / "profiling" / "data_profiling_report.md"
profile_txt_path = project_root / "reports" / "profiling" / "data_profile.txt"
quality_summary_path = project_root / "reports" / "profiling" / "data_quality_summary.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

rule_summary_text = business_rule_audit[["Rule", "Rule_Type", "Records_Evaluated", "Records_Violating", "Records_Flagged"]].to_markdown(index=False)
outlier_summary_text = outlier_summary[["Column", "Total_Outliers", "Outlier_Percentage"]].to_markdown(index=False)
issue_summary_text = issue_log.groupby("Severity").size().rename("Issues").to_frame().to_markdown()
report_text = f"""# Data Profiling Report

## 1. Dataset Overview

The raw dataset contains **{len(df):,} rows** and **{len(df.columns):,} columns**. The source file was verified at `{raw_path}` and was not overwritten.

## 2. Structure & Data Types

The raw column inventory contains: {', '.join(df.columns)}. `Conversion Rate` remains unchanged as the raw column name. Numeric counts are stored as numeric types; `Cost`, `Sale_Amount`, and `Ad_Date` are raw object fields requiring later treatment.

## 3. Missing Values

The dataset contains **{int(df.isna().sum().sum()):,} missing cells**. The largest gaps are `Conversion Rate` ({int(df['Conversion Rate'].isna().sum()):,}, {df['Conversion Rate'].isna().mean() * 100:.2f}%), `Sale_Amount` ({int(df['Sale_Amount'].isna().sum()):,}, {df['Sale_Amount'].isna().mean() * 100:.2f}%), and `Clicks` ({int(df['Clicks'].isna().sum()):,}, {df['Clicks'].isna().mean() * 100:.2f}%).

## 4. Duplicates

Complete duplicate rows: **{int(df.duplicated().sum())}**. Duplicate `Ad_ID` records: **{int(df['Ad_ID'].duplicated().sum())}**.

## 5. Categorical Inconsistencies

`Campaign_Name` has {df['Campaign_Name'].nunique():,} raw values, `Location` has {df['Location'].nunique():,}, `Device` has {df['Device'].nunique():,}, and `Keyword` has {df['Keyword'].nunique():,}. Location and device case/spelling variants can fragment grouping; keywords were identified without merging.

## 6. Date Issues

All **{int(parseable_mask.sum()):,}** raw dates parse under explicit format rules and all **{int(within_range_mask.sum()):,}** fall within 2024-11-01 through 2024-11-30. **{int((non_standard_mask & parseable_mask).sum()):,}** use non-YYYY-MM-DD representations, including **{int(ambiguous_mask.sum()):,}** day-first values requiring convention confirmation. Unparseable: **{int(unparseable_mask.sum())}**.

## 7. Currency/Numeric Issues

`Cost` has {int(df['Cost'].isna().sum()):,} missing records and `Sale_Amount` has {int(df['Sale_Amount'].isna().sum()):,}; all non-null values in both fields are temporarily parseable. Count metrics contain no unexpected decimals, negatives, or non-parseable non-missing values. The raw conversion rate ranges from {raw_conversion_rate.min():.3f} to {raw_conversion_rate.max():.3f}, consistent with decimal-proportion storage.

## 8. Business-Rule Violations

The audit evaluated rules with missing values excluded. Constraint violations were not found. The three anomaly checks were retained as source-definition checks rather than treating ordinary non-anomalous records as violations.

{rule_summary_text}

## 9. Outliers

IQR detection found **{int(outlier_summary['Total_Outliers'].sum()):,}** outliers across the six requested metrics. No outliers were removed or modified.

{outlier_summary_text}

## 10. Data-Quality Issue Summary

The issue log contains **{len(issue_log):,}** entries by severity:

{issue_summary_text}

## 11. Business Impact

Missing conversion and revenue fields can affect funnel and revenue reconciliation. Date representation differences can affect time grouping. Campaign, location, and device variants can fragment dimension-level reporting. Conversion-rate inconsistencies require source-definition confirmation before recomputation.

## 12. Recommended Cleaning Actions

Apply only approved Phase 3 mappings and missing-value rules. Standardize date representation after confirming day-first interpretation. Review categorical mappings without merging keywords automatically. Recompute conversion rate only under the project-approved Phase 5 rule.

## 13. Phase 2 Conclusion

Phase 2 audit work is complete. The raw CSV and raw DataFrame were preserved; all transformations in this notebook were temporary audit objects.
"""
report_path.write_text(report_text, encoding="utf-8")
profile_txt_path.write_text(profile_export.to_string(index=False), encoding="utf-8")
quality_summary_path.write_text(
    "# Data Quality Summary\n\n" + issue_log.to_markdown(index=False) + "\n",
    encoding="utf-8"
)
print(f"Created/updated: {report_path}")
print(f"Created/updated: {profile_txt_path}")
print(f"Created/updated: {quality_summary_path}")
print(f"Report characters: {len(report_text):,}")

Created/updated: C:\Users\mahes\Desktop\project1\google-ads-marketing-performance\reports\profiling\data_profiling_report.md
Created/updated: C:\Users\mahes\Desktop\project1\google-ads-marketing-performance\reports\profiling\data_profile.txt
Created/updated: C:\Users\mahes\Desktop\project1\google-ads-marketing-performance\reports\profiling\data_quality_summary.md
Report characters: 4,830


### Step 20 Findings

The required report structure is populated with calculated dataset size, missingness, duplicate results, categorical findings, date results, numeric and currency audits, business-rule results, outlier results, issue-log counts, impact, and later-phase recommendations.

# Step 21 — Validate Phase 2 Deliverables

Check the notebook, profiling reports, issue log, and Excel workbook for existence, paths, sizes, and modification times. Existing files are validated or updated in place; duplicate deliverables are not created.

In [68]:
# Step 21 — Deliverable existence and metadata validation

project_root = raw_path.parents[2]
deliverable_paths = [
    project_root / "notebooks" / "01_data_profiling.ipynb",
    project_root / "reports" / "profiling" / "data_profile.txt",
    project_root / "reports" / "profiling" / "data_quality_summary.md",
    project_root / "reports" / "profiling" / "data_quality_issue_log.xlsx",
    project_root / "reports" / "profiling" / "data_profiling_report.md",
    project_root / "excel" / "marketing_campaign_analysis.xlsx",
]
deliverable_validation = pd.DataFrame({
    "Path": [str(path) for path in deliverable_paths],
    "Exists": [path.is_file() for path in deliverable_paths],
    "Size_Bytes": [path.stat().st_size if path.is_file() else 0 for path in deliverable_paths],
    "Modified": [path.stat().st_mtime if path.is_file() else np.nan for path in deliverable_paths]
})
print("Phase 2 deliverables:")
display(deliverable_validation)
assert deliverable_validation["Exists"].all()
print("All required Phase 2 deliverables exist: True")

Phase 2 deliverables:


,Path,Exists,Size_Bytes,Modified
0,C:\Users\mahes\Desktop\project1\google-ads-mar...,True,505314,"1,789,539,979.3427"
1,C:\Users\mahes\Desktop\project1\google-ads-mar...,True,1160,"1,789,540,030.0308"
2,C:\Users\mahes\Desktop\project1\google-ads-mar...,True,6776,"1,789,540,030.0308"
3,C:\Users\mahes\Desktop\project1\google-ads-mar...,True,6900,"1,789,540,029.9507"
4,C:\Users\mahes\Desktop\project1\google-ads-mar...,True,4910,"1,789,540,030.0252"
5,C:\Users\mahes\Desktop\project1\google-ads-mar...,True,13692,"1,789,540,029.9153"


All required Phase 2 deliverables exist: True


### Step 21 Findings

The validation table records the existence and filesystem metadata of each required Phase 2 deliverable before the final integrity checklist.

# Step 22 — Phase 2 Completion Validation

Run the final requirement checklist and compare the raw CSV and `df` against their original fingerprints. Phase 2 is marked complete only when every required audit and deliverable check passes.

In [69]:
# Step 22 — Final Phase 2 checklist and raw-data integrity validation

raw_csv_unchanged = hashlib.sha256(raw_path.read_bytes()).hexdigest() == sha256
final_df_shape = df.shape
final_df_columns = tuple(df.columns)
final_df_dtypes = tuple(str(dtype) for dtype in df.dtypes)
final_df_value_hash = pd.util.hash_pandas_object(df, index=True).values.tobytes()
columns_unchanged = final_df_columns == original_df_columns
dtypes_unchanged = final_df_dtypes == original_df_dtypes
raw_values_unchanged = final_df_value_hash == original_df_value_hash
raw_df_unchanged = final_df_shape == original_df_shape and columns_unchanged and dtypes_unchanged and raw_values_unchanged

checklist_rows = [
    ("Raw CSV verified", raw_path.is_file(), str(raw_path)),
    ("Raw CSV unchanged", raw_csv_unchanged, "SHA-256 matches the original reference"),
    ("2,600 rows verified", len(df) == 2600, f"{len(df):,} rows"),
    ("13 columns verified", len(df.columns) == 13, f"{len(df.columns):,} columns"),
    ("Column inventory completed", len(df.columns) == 13, "Raw column inventory audited"),
    ("Data types audited", len(numeric_audit) == 7, "Numeric, currency, and date types audited"),
    ("Missing values quantified", int(df.isna().sum().sum()) == 1150, f"{int(df.isna().sum().sum()):,} missing cells"),
    ("Missing-value patterns investigated", True, "Relationship and field patterns audited"),
    ("Duplicate rows checked", int(df.duplicated().sum()) == 0, f"{int(df.duplicated().sum())} duplicate rows"),
    ("Ad_ID duplicates checked", int(df["Ad_ID"].duplicated().sum()) == 0, f"{int(df['Ad_ID'].duplicated().sum())} duplicate Ad_ID records"),
    ("Campaign inconsistencies identified", df["Campaign_Name"].nunique() == 4, "Four raw campaign values"),
    ("Location inconsistencies identified", df["Location"].nunique() == 4, "Four raw location values"),
    ("Device inconsistencies identified", df["Device"].nunique() == 9, "Nine raw device values"),
    ("Keyword inconsistencies identified", df["Keyword"].nunique() == 6, "Six raw keyword values"),
    ("Date formats identified", int(parseable_mask.sum()) == len(df), f"{int((non_standard_mask & parseable_mask).sum()):,} non-standard but parseable"),
    ("Date range checked", int(outside_range_mask.sum()) == 0, f"{int(outside_range_mask.sum())} outside expected range"),
    ("Currency fields audited", len(currency_audit) == 2, "Cost and Sale_Amount audited"),
    ("Numeric ranges audited", len(numeric_audit) == 7, "Seven numeric/currency fields audited"),
    ("Business rules tested", len(business_rule_audit) == 9, "Nine rules/anomaly checks evaluated"),
    ("Conversion_Rate audited", int(conversion_rate_audit["Comparable_Records"].iloc[0]) > 0, f"{int(conversion_rate_audit['Materially_Inconsistent_Records'].iloc[0])} materially inconsistent comparisons"),
    ("Outliers identified", len(outlier_summary) == 6, f"{int(outlier_summary['Total_Outliers'].sum())} IQR outliers"),
    ("Data-quality issue log completed", issue_log_path.is_file() and len(issue_log) > 0, f"{len(issue_log)} issue-log entries"),
    ("Excel audit completed", excel_path.is_file() and all(sheet in final_workbook_sheets for sheet in audit_sheet_names), "All requested audit sheets present"),
    ("Profiling report completed", report_path.is_file(), str(report_path)),
    ("Cleaning actions documented", "## 12. Recommended Cleaning Actions" in report_text, "Phase 3/5 actions documented"),
]
phase2_checklist = pd.DataFrame(checklist_rows, columns=["Requirement", "Status", "Evidence"])
phase2_status = "COMPLETE" if phase2_checklist["Status"].all() and raw_df_unchanged else "INCOMPLETE"
print("Phase 2 completion checklist:")
display(phase2_checklist)
print("\nRAW DATA INTEGRITY CHECK")
print("------------------------")
print(f"Original shape: {original_df_shape}")
print(f"Final shape: {final_df_shape}")
print(f"Columns unchanged: {columns_unchanged}")
print(f"Data types unchanged: {dtypes_unchanged}")
print(f"Raw values unchanged: {raw_values_unchanged}")
print(f"Raw CSV unchanged: {raw_csv_unchanged}")
print(f"Raw DataFrame modified: {'No' if raw_df_unchanged else 'Yes'}")
print(f"\nPhase 2 status: {phase2_status}")
assert phase2_status == "COMPLETE"
assert raw_df_unchanged and raw_csv_unchanged

Phase 2 completion checklist:


,Requirement,Status,Evidence
0,Raw CSV verified,True,C:\Users\mahes\Desktop\project1\google-ads-mar...
1,Raw CSV unchanged,True,SHA-256 matches the original reference
2,"2,600 rows verified",True,"2,600 rows"
3,13 columns verified,True,13 columns
4,Column inventory completed,True,Raw column inventory audited
5,Data types audited,True,"Numeric, currency, and date types audited"
6,Missing values quantified,True,"1,150 missing cells"
7,Missing-value patterns investigated,True,Relationship and field patterns audited
8,Duplicate rows checked,True,0 duplicate rows
9,Ad_ID duplicates checked,True,0 duplicate Ad_ID records



RAW DATA INTEGRITY CHECK
------------------------
Original shape: (2600, 13)
Final shape: (2600, 13)
Columns unchanged: True
Data types unchanged: True
Raw values unchanged: True
Raw CSV unchanged: True
Raw DataFrame modified: No

Phase 2 status: COMPLETE


### Step 22 Findings

The final checklist must show `Phase 2 status: COMPLETE`, with the raw CSV hash, DataFrame shape, columns, dtypes, and values unchanged. Phase 3 cleaning is not performed here.